# Figure 3 | Normal striatal MSN spatial gradients

Cell-type UMAPs, tissue maps, S/M scores, DC1 correlations, and WT anatomical gradients.

Check input/output paths and settings before running each workflow. Run its cells in order with a fresh kernel. See README.md for required inputs, panel coverage, and configuration differences.

| Panels | Workflow |
|---|---|
| A | Major-cell-type UMAP |
| B-F | Refined UMAPs, tissue maps, scores, and DC1 |
| G | DC1-score correlations |
| I, BASSFISH maps | Reusable single-gene spatial map routine |
| J-K | WT-reference heatmaps and category profiles |
| H; I, Allen images | Schematics and external reference images |


## Panel A | Major-cell-type UMAP


In [ ]:
# Imports and settings.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl


In [ ]:
H5AD_PATH = r"F:\BASSFISH_HD\Source_data\integrated_str_final_with_D1_D2_DC.h5ad"

OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\major_cluster_4_umap.pdf"


CELLTYPE_COL = "major_cluster"
UMAP_KEY = "X_umap"


FIG_WIDTH_MM = 55
FIG_HEIGHT_MM = 50


POINT_SIZE = 0.06
POINT_ALPHA = 0.90


SHOW_LEGEND = True


LEGEND_OUTSIDE = True

LEGEND_FONTSIZE = 7
LEGEND_MARKERSCALE = 6


LEGEND_LABELSPACING = 0.35


DPI = 600

SHOW_TITLE = False
TITLE = None

TRANSPARENT = True


In [ ]:
refined_cluster_4_cmap = {
    'STR_neuron': '#E618FF',
    'Oligo': '#EB6891',
    'OPC': '#A584F7',
    'Astrocytes': '#006837',
    'Mircrogila': '#FAD6EA',
    'Vascular_endo': '#DD8B59',
    'Pericytes': '#8A0179',
}


In [ ]:
PLOT_ORDER = [

    "Oligo",
    "OPC",
    "Astrocytes",
    "Mircrogila",
    "Vascular_endo",
    "Pericytes",


    "STR_neuron",
]


In [ ]:
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,

    "axes.titlesize": 7,
    "axes.labelsize": 7,

    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    "legend.fontsize": 7,


    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.linewidth": 0.5,
})


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)

print(adata)

if CELLTYPE_COL not in adata.obs.columns:
    raise KeyError(
        f"{CELLTYPE_COL!r} is missing from adata.obs.\n"
        f"Available obs columns:\n{adata.obs.columns.tolist()}"
    )

if UMAP_KEY not in adata.obsm:
    raise KeyError(
        f"{UMAP_KEY!r} is missing from adata.obsm.\n"
        f"Available obsm keys: {list(adata.obsm.keys())}"
    )

print("\nCell-type counts:")
print(
    adata.obs[CELLTYPE_COL]
    .astype(str)
    .value_counts()
)


In [ ]:
celltypes = (
    adata.obs[CELLTYPE_COL]
    .astype(str)
)

present_celltypes = list(
    pd.unique(celltypes)
)

missing_colors = [
    ct
    for ct in present_celltypes
    if ct not in refined_cluster_4_cmap
]

unused_colors = [
    ct
    for ct in refined_cluster_4_cmap
    if ct not in present_celltypes
]

print("Number of cell types in data:", len(present_celltypes))

if len(missing_colors) > 0:
    print("\nWARNING: The following cell types have no assigned colors and will use gray #BDBDBD:")
    for ct in missing_colors:
        print("  ", ct)
else:
    print("\nAll cell types in the data have assigned colors.")

if len(unused_colors) > 0:
    print("\nCategories in the color dictionary that are absent from the data:")
    for ct in unused_colors:
        print("  ", ct)


In [ ]:
def plot_celltype_umap(
    adata,
    celltype_col="refined_cluster_4",
    umap_key="X_umap",
    color_dict=None,
    plot_order=None,

    fig_width_mm=55,
    fig_height_mm=50,

    point_size=0.08,
    point_alpha=0.90,

    show_legend=True,
    legend_outside=True,
    legend_fontsize=7,
    legend_markerscale=6,
    legend_labelspacing=0.35,

    show_title=False,
    title=None,

    missing_color="#BDBDBD",

    save=None,
    dpi=600,
    transparent=True,
):
    """Plot UMAP coordinates colored by cell type."""


    if color_dict is None:
        raise ValueError("color_dict is required.")

    if plot_order is None:
        plot_order = list(color_dict.keys())


    umap = np.asarray(
        adata.obsm[umap_key]
    )

    x = umap[:, 0]
    y = umap[:, 1]

    labels = (
        adata.obs[celltype_col]
        .astype(str)
        .to_numpy()
    )


    present = set(labels)

    ordered_categories = [
        ct
        for ct in plot_order
        if ct in present
    ]

    extra_categories = sorted([
        ct
        for ct in present
        if ct not in plot_order
    ])

    categories = (
        extra_categories
        +
        ordered_categories
    )


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )


    for ct in categories:

        mask = (
            labels == ct
        )

        color = color_dict.get(
            ct,
            missing_color
        )

        ax.scatter(
            x[mask],
            y[mask],

            s=point_size,
            c=color,

            alpha=point_alpha,

            linewidths=0,

            rasterized=True,

            label=ct,
        )


    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="datalim"
    )


    if show_title:

        if title is None:
            title = celltype_col

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    if show_legend:

        if legend_outside:

            legend = ax.legend(
                frameon=False,

                loc="center left",
                bbox_to_anchor=(1.01, 0.5),

                fontsize=legend_fontsize,

                markerscale=legend_markerscale,

                handletextpad=0.25,
                borderaxespad=0,
                borderpad=0,

                labelspacing=legend_labelspacing,
            )

        else:

            legend = ax.legend(
                frameon=False,
                loc="best",

                fontsize=legend_fontsize,
                markerscale=legend_markerscale,

                handletextpad=0.25,
                borderpad=0,
                labelspacing=legend_labelspacing,
            )


        legend.set_rasterized(False)


    if save is not None:

        save = Path(save)

        if save.parent != Path("."):
            save.parent.mkdir(
                parents=True,
                exist_ok=True
            )

        fig.savefig(
            save,

            dpi=dpi,

            bbox_inches="tight",
            pad_inches=0.02,

            transparent=transparent,
        )

        print(
            f"Saved:\n{save.resolve()}"
        )

    plt.show()

    return fig, ax


In [ ]:
fig, ax = plot_celltype_umap(
    adata,

    celltype_col=CELLTYPE_COL,
    umap_key=UMAP_KEY,

    color_dict=refined_cluster_4_cmap,
    plot_order=PLOT_ORDER,


    fig_width_mm=FIG_WIDTH_MM,
    fig_height_mm=FIG_HEIGHT_MM,


    point_size=POINT_SIZE,
    point_alpha=POINT_ALPHA,


    show_legend=SHOW_LEGEND,
    legend_outside=LEGEND_OUTSIDE,

    legend_fontsize=LEGEND_FONTSIZE,
    legend_markerscale=LEGEND_MARKERSCALE,
    legend_labelspacing=LEGEND_LABELSPACING,


    show_title=SHOW_TITLE,
    title=TITLE,


    save=OUTPUT_PDF,

    dpi=DPI,
    transparent=TRANSPARENT,
)


## Panels B-F | MSN subclusters, tissue maps, scores, and DC1

Run refined UMAPs, score maps, highlighted dorsal UMAP, DC1 maps, and cell-type tissue maps in order. Inputs must contain UMAP coordinates, subcluster labels, S/M scores, and diffusion components. Upstream clustering, gene-set scoring, and diffusion-map fitting are not included.


In [ ]:
# Imports and settings.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl


In [ ]:
H5AD_PATH = r"F:\BASSFISH_HD\Source_data\integrated_str_final_with_D1_D2_DC.h5ad"

OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\refined_cluster_4_umap.pdf"


CELLTYPE_COL = "refined_cluster_4"
UMAP_KEY = "X_umap"


FIG_WIDTH_MM = 55
FIG_HEIGHT_MM = 50


POINT_SIZE = 0.05
POINT_ALPHA = 0.90


SHOW_LEGEND = True


LEGEND_OUTSIDE = True

LEGEND_FONTSIZE = 7
LEGEND_MARKERSCALE = 6


LEGEND_LABELSPACING = 0.35


DPI = 600

SHOW_TITLE = False
TITLE = None

TRANSPARENT = True


In [ ]:
refined_cluster_4_cmap = {
    'Dorsal_M_D1': '#F7EBA7',
    'Dorsal_M_D2': '#a0d4a5',
    'Dorsal_S_D1': '#061584',
    'Dorsal_S_D2': '#B34CFF',
    'NAc_D1': '#FFA900',
    'NAc_D2': '#5cb998',
    'Ventral_D1': '#EF6702',
    'Ventral_D2': '#006837',
    'D1H_Nxph4': '#70F1FC',
    'D1H_Cpne7': '#93C9ED',
    'STR_chat': '#C1A3CE',
    'STR_Rreb1_neuron': '#742981',

    'Oligo': '#EB6891',
    'OPC': '#A584F7',
    'Astrocytes': '#006837',
    'Mircrogila': '#FAD6EA',
    'Vascular_endo': '#DD8B59',
    'Pericytes': '#8A0179',
}


In [ ]:
PLOT_ORDER = [

    "Oligo",
    "OPC",
    "Astrocytes",
    "Mircrogila",
    "Vascular_endo",
    "Pericytes",


    "STR_chat",
    "STR_Rreb1_neuron",
    "D1H_Nxph4",
    "D1H_Cpne7",


    "Ventral_D1",
    "NAc_D1",
    "Dorsal_S_D1",
    "NAc_D2",
    "Dorsal_S_D2",
    "Dorsal_M_D1",
    "Dorsal_M_D2",
    "Ventral_D2",
]


In [ ]:
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,

    "axes.titlesize": 7,
    "axes.labelsize": 7,

    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    "legend.fontsize": 7,


    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.linewidth": 0.5,
})


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)

print(adata)

if CELLTYPE_COL not in adata.obs.columns:
    raise KeyError(
        f"{CELLTYPE_COL!r} is missing from adata.obs.\n"
        f"Available obs columns:\n{adata.obs.columns.tolist()}"
    )

if UMAP_KEY not in adata.obsm:
    raise KeyError(
        f"{UMAP_KEY!r} is missing from adata.obsm.\n"
        f"Available obsm keys: {list(adata.obsm.keys())}"
    )

print("\nCell-type counts:")
print(
    adata.obs[CELLTYPE_COL]
    .astype(str)
    .value_counts()
)


In [ ]:
celltypes = (
    adata.obs[CELLTYPE_COL]
    .astype(str)
)

present_celltypes = list(
    pd.unique(celltypes)
)

missing_colors = [
    ct
    for ct in present_celltypes
    if ct not in refined_cluster_4_cmap
]

unused_colors = [
    ct
    for ct in refined_cluster_4_cmap
    if ct not in present_celltypes
]

print("Number of cell types in data:", len(present_celltypes))

if len(missing_colors) > 0:
    print("\nWARNING: The following cell types have no assigned colors and will use gray #BDBDBD:")
    for ct in missing_colors:
        print("  ", ct)
else:
    print("\nAll cell types in the data have assigned colors.")

if len(unused_colors) > 0:
    print("\nCategories in the color dictionary that are absent from the data:")
    for ct in unused_colors:
        print("  ", ct)


In [ ]:
def plot_celltype_umap(
    adata,
    celltype_col="refined_cluster_4",
    umap_key="X_umap",
    color_dict=None,
    plot_order=None,

    fig_width_mm=55,
    fig_height_mm=50,

    point_size=0.08,
    point_alpha=0.90,

    show_legend=True,
    legend_outside=True,
    legend_fontsize=7,
    legend_markerscale=6,
    legend_labelspacing=0.35,

    show_title=False,
    title=None,

    missing_color="#BDBDBD",

    save=None,
    dpi=600,
    transparent=True,
):
    """Plot UMAP coordinates colored by cell type."""


    if color_dict is None:
        raise ValueError("color_dict is required.")

    if plot_order is None:
        plot_order = list(color_dict.keys())


    umap = np.asarray(
        adata.obsm[umap_key]
    )

    x = umap[:, 0]
    y = umap[:, 1]

    labels = (
        adata.obs[celltype_col]
        .astype(str)
        .to_numpy()
    )


    present = set(labels)

    ordered_categories = [
        ct
        for ct in plot_order
        if ct in present
    ]

    extra_categories = sorted([
        ct
        for ct in present
        if ct not in plot_order
    ])

    categories = (
        extra_categories
        +
        ordered_categories
    )


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )


    for ct in categories:

        mask = (
            labels == ct
        )

        color = color_dict.get(
            ct,
            missing_color
        )

        ax.scatter(
            x[mask],
            y[mask],

            s=point_size,
            c=color,

            alpha=point_alpha,

            linewidths=0,

            rasterized=True,

            label=ct,
        )


    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="datalim"
    )


    if show_title:

        if title is None:
            title = celltype_col

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    if show_legend:

        if legend_outside:

            legend = ax.legend(
                frameon=False,

                loc="center left",
                bbox_to_anchor=(1.01, 0.5),

                fontsize=legend_fontsize,

                markerscale=legend_markerscale,

                handletextpad=0.25,
                borderaxespad=0,
                borderpad=0,

                labelspacing=legend_labelspacing,
            )

        else:

            legend = ax.legend(
                frameon=False,
                loc="best",

                fontsize=legend_fontsize,
                markerscale=legend_markerscale,

                handletextpad=0.25,
                borderpad=0,
                labelspacing=legend_labelspacing,
            )


        legend.set_rasterized(False)


    if save is not None:

        save = Path(save)

        if save.parent != Path("."):
            save.parent.mkdir(
                parents=True,
                exist_ok=True
            )

        fig.savefig(
            save,

            dpi=dpi,

            bbox_inches="tight",
            pad_inches=0.02,

            transparent=transparent,
        )

        print(
            f"Saved:\n{save.resolve()}"
        )

    plt.show()

    return fig, ax


In [ ]:
fig, ax = plot_celltype_umap(
    adata,

    celltype_col=CELLTYPE_COL,
    umap_key=UMAP_KEY,

    color_dict=refined_cluster_4_cmap,
    plot_order=PLOT_ORDER,


    fig_width_mm=FIG_WIDTH_MM,
    fig_height_mm=FIG_HEIGHT_MM,


    point_size=POINT_SIZE,
    point_alpha=POINT_ALPHA,


    show_legend=SHOW_LEGEND,
    legend_outside=LEGEND_OUTSIDE,

    legend_fontsize=LEGEND_FONTSIZE,
    legend_markerscale=LEGEND_MARKERSCALE,
    legend_labelspacing=LEGEND_LABELSPACING,


    show_title=SHOW_TITLE,
    title=TITLE,


    save=OUTPUT_PDF,

    dpi=DPI,
    transparent=TRANSPARENT,
)


In [ ]:
DORSAL_CLUSTER_PREFIX = "Dorsal_"

S_SCORE_COL = "SM_S_score"
M_SCORE_COL = "SM_M_score"


NORMALIZE_SCORE_01 = True


SCORE_FIG_WIDTH_MM = 52
SCORE_FIG_HEIGHT_MM = 48


SCORE_BG_POINT_SIZE = 0.05
SCORE_FG_POINT_SIZE = 0.10

SCORE_BG_ALPHA = 0.30
SCORE_FG_ALPHA = 0.95


SCORE_CMAP = "seismic"


SCORE_VMIN = 0.2
SCORE_VMAX = 1.00

BACKGROUND_COLOR = "#D0D0D0"


S_SCORE_OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\Dorsal_S_score_umap.pdf"
M_SCORE_OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\Dorsal_M_score_umap.pdf"


In [ ]:
def minmax_01(x):
    """Scale finite values to 0-1 and retain missing values."""


    x = np.asarray(x, dtype=float)

    out = np.full(
        x.shape,
        np.nan,
        dtype=float
    )

    valid = np.isfinite(x)

    if valid.sum() == 0:
        return out

    xmin = np.nanmin(x[valid])
    xmax = np.nanmax(x[valid])

    if np.isclose(xmax, xmin):
        out[valid] = 0.5
    else:
        out[valid] = (
            x[valid] - xmin
        ) / (
            xmax - xmin
        )

    return out


In [ ]:
def plot_dorsal_score_umap(
    adata,
    score_col,

    celltype_col="refined_cluster_4",
    dorsal_prefix="Dorsal_",
    umap_key="X_umap",

    normalize_01=True,

    cmap="viridis",
    color_vmin=0.30,
    color_vmax=1.00,

    background_color="#D0D0D0",

    fig_width_mm=52,
    fig_height_mm=48,

    bg_point_size=0.05,
    fg_point_size=0.10,

    bg_alpha=0.30,
    fg_alpha=0.95,

    title=None,
    show_title=False,

    colorbar_label=None,

    save=None,
    dpi=600,
    transparent=True,
):
    """Plot dorsal MSN scores on the striatal UMAP."""


    if score_col not in adata.obs.columns:
        raise KeyError(
            f"{score_col!r} is missing from adata.obs."
        )

    if celltype_col not in adata.obs.columns:
        raise KeyError(
            f"{celltype_col!r} is missing from adata.obs."
        )

    if umap_key not in adata.obsm:
        raise KeyError(
            f"{umap_key!r} is missing from adata.obsm."
        )


    umap = np.asarray(
        adata.obsm[umap_key]
    )

    x = umap[:, 0]
    y = umap[:, 1]


    celltypes = (
        adata.obs[celltype_col]
        .astype(str)
        .to_numpy()
    )

    dorsal_mask = np.array([
        str(v).startswith(dorsal_prefix)
        for v in celltypes
    ])


    score = pd.to_numeric(
        adata.obs[score_col],
        errors="coerce"
    ).to_numpy(dtype=float)


    fg_mask = (
        dorsal_mask
        &
        np.isfinite(score)
    )


    bg_mask = ~fg_mask

    print("=" * 60)
    print("Score:", score_col)
    print(f"All cells: {adata.n_obs:,}")
    print(f"Dorsal cells: {dorsal_mask.sum():,}")
    print(f"Dorsal cells with valid score: {fg_mask.sum():,}")
    print(f"Gray background cells: {bg_mask.sum():,}")


    plot_score = score.copy()

    if normalize_01:
        plot_score[fg_mask] = minmax_01(
            score[fg_mask]
        )


        vmin = color_vmin
        vmax = color_vmax

    else:

        if color_vmin is None:
            vmin = np.nanmin(
                score[fg_mask]
            )
        else:
            vmin = color_vmin

        if color_vmax is None:
            vmax = np.nanmax(
                score[fg_mask]
            )
        else:
            vmax = color_vmax


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )


    ax.scatter(
        x[bg_mask],
        y[bg_mask],

        s=bg_point_size,
        c=background_color,

        alpha=bg_alpha,

        linewidths=0,

        rasterized=True,
    )


    fg_indices = np.where(
        fg_mask
    )[0]

    order = np.argsort(
        plot_score[fg_indices]
    )

    fg_indices = fg_indices[
        order
    ]

    im = ax.scatter(
        x[fg_indices],
        y[fg_indices],

        s=fg_point_size,

        c=plot_score[fg_indices],

        cmap=cmap,
        vmin=vmin,
        vmax=vmax,

        alpha=fg_alpha,

        linewidths=0,

        rasterized=True,
    )


    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="datalim"
    )


    if show_title:

        if title is None:
            title = score_col

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    cbar = fig.colorbar(
        im,
        ax=ax,

        fraction=0.035,
        pad=0.02,
    )

    if colorbar_label is None:

        if normalize_01:
            colorbar_label = (
                f"{score_col} (0–1)"
            )
        else:
            colorbar_label = score_col

    cbar.set_label(
        colorbar_label,
        fontsize=7,
        labelpad=2,
    )

    cbar.ax.tick_params(
        labelsize=7,
        width=0.5,
        length=2,
        pad=1,
    )

    cbar.outline.set_linewidth(
        0.5
    )


    if save is not None:

        save = Path(save)

        if save.parent != Path("."):
            save.parent.mkdir(
                parents=True,
                exist_ok=True
            )

        fig.savefig(
            save,

            dpi=dpi,

            bbox_inches="tight",
            pad_inches=0.02,

            transparent=transparent,
        )

        print(
            f"Saved:\n{save.resolve()}"
        )

    plt.show()

    return fig, ax


In [ ]:
fig_s, ax_s = plot_dorsal_score_umap(
    adata,

    score_col=S_SCORE_COL,

    celltype_col=CELLTYPE_COL,
    dorsal_prefix=DORSAL_CLUSTER_PREFIX,

    umap_key=UMAP_KEY,

    normalize_01=NORMALIZE_SCORE_01,

    cmap=SCORE_CMAP,
    color_vmin=SCORE_VMIN,
    color_vmax=SCORE_VMAX,

    background_color=BACKGROUND_COLOR,

    fig_width_mm=SCORE_FIG_WIDTH_MM,
    fig_height_mm=SCORE_FIG_HEIGHT_MM,

    bg_point_size=SCORE_BG_POINT_SIZE,
    fg_point_size=SCORE_FG_POINT_SIZE,

    bg_alpha=SCORE_BG_ALPHA,
    fg_alpha=SCORE_FG_ALPHA,

    show_title=False,
    title="Dorsal S-score",

    colorbar_label=(
        "S-score (0–1)"
        if NORMALIZE_SCORE_01
        else "S-score"
    ),

    save=S_SCORE_OUTPUT_PDF,

    dpi=DPI,
    transparent=TRANSPARENT,
)


In [ ]:
fig_m, ax_m = plot_dorsal_score_umap(
    adata,

    score_col=M_SCORE_COL,

    celltype_col=CELLTYPE_COL,
    dorsal_prefix=DORSAL_CLUSTER_PREFIX,

    umap_key=UMAP_KEY,

    normalize_01=NORMALIZE_SCORE_01,

    cmap=SCORE_CMAP,
    color_vmin=SCORE_VMIN,
    color_vmax=SCORE_VMAX,

    background_color=BACKGROUND_COLOR,

    fig_width_mm=SCORE_FIG_WIDTH_MM,
    fig_height_mm=SCORE_FIG_HEIGHT_MM,

    bg_point_size=SCORE_BG_POINT_SIZE,
    fg_point_size=SCORE_FG_POINT_SIZE,

    bg_alpha=SCORE_BG_ALPHA,
    fg_alpha=SCORE_FG_ALPHA,

    show_title=False,
    title="Dorsal M-score",

    colorbar_label=(
        "M-score (0–1)"
        if NORMALIZE_SCORE_01
        else "M-score"
    ),

    save=M_SCORE_OUTPUT_PDF,

    dpi=DPI,
    transparent=TRANSPARENT,
)


In [ ]:
DORSAL_OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\Dorsal_clusters_umap.pdf"

DORSAL_FIG_WIDTH_MM = 55
DORSAL_FIG_HEIGHT_MM = 50

DORSAL_BG_POINT_SIZE = 0.2
DORSAL_FG_POINT_SIZE = 0.10

DORSAL_BG_ALPHA = 0.25
DORSAL_FG_ALPHA = 0.95

DORSAL_BG_COLOR = "#D0D0D0"


DORSAL_PLOT_ORDER = [


    "Dorsal_S_D1",
    "Dorsal_M_D1",

    "Dorsal_M_D2",
    "Dorsal_S_D2",

]


umap = np.asarray(
    adata.obsm[UMAP_KEY]
)

x = umap[:, 0]
y = umap[:, 1]

labels = (
    adata.obs[CELLTYPE_COL]
    .astype(str)
    .to_numpy()
)


dorsal_mask = np.array([
    label.startswith("Dorsal_")
    for label in labels
])

print(f"All cells: {adata.n_obs:,}")
print(f"Dorsal cells: {dorsal_mask.sum():,}")

print("\nDorsal cluster counts:")
print(
    adata.obs.loc[
        dorsal_mask,
        CELLTYPE_COL
    ]
    .astype(str)
    .value_counts()
)


mm_to_inch = 1 / 25.4

fig, ax = plt.subplots(
    figsize=(
        DORSAL_FIG_WIDTH_MM * mm_to_inch,
        DORSAL_FIG_HEIGHT_MM * mm_to_inch,
    )
)


ax.scatter(
    x[~dorsal_mask],
    y[~dorsal_mask],

    s=DORSAL_BG_POINT_SIZE,
    c=DORSAL_BG_COLOR,

    alpha=DORSAL_BG_ALPHA,

    linewidths=0,

    rasterized=True,
)


for cluster in DORSAL_PLOT_ORDER:

    mask = (
        labels == cluster
    )

    if mask.sum() == 0:
        print(
            f"Warning: {cluster} has no cells"
        )
        continue

    color = refined_cluster_4_cmap[
        cluster
    ]

    ax.scatter(
        x[mask],
        y[mask],

        s=DORSAL_FG_POINT_SIZE,

        c=color,

        alpha=DORSAL_FG_ALPHA,

        linewidths=0,

        rasterized=True,

        label=cluster,
    )


ax.set_xticks([])
ax.set_yticks([])

ax.set_xlabel("")
ax.set_ylabel("")

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_aspect(
    "equal",
    adjustable="datalim"
)


legend = ax.legend(
    frameon=False,

    loc="center left",
    bbox_to_anchor=(1.01, 0.5),

    fontsize=7,

    markerscale=6,

    handletextpad=0.25,
    borderaxespad=0,
    borderpad=0,

    labelspacing=0.35,
)

legend.set_rasterized(False)


fig.savefig(
    DORSAL_OUTPUT_PDF,

    dpi=600,

    bbox_inches="tight",
    pad_inches=0.02,

    transparent=True,
)

plt.show()


In [ ]:
def _minmax_01_spatial(x):
    x = np.asarray(x, dtype=float)

    out = np.full(
        x.shape,
        np.nan,
        dtype=float
    )

    valid = np.isfinite(x)

    if valid.sum() == 0:
        return out

    xmin = np.nanmin(x[valid])
    xmax = np.nanmax(x[valid])

    if np.isclose(xmin, xmax):
        out[valid] = 0.5
    else:
        out[valid] = (
            x[valid] - xmin
        ) / (
            xmax - xmin
        )

    return out


def _transform_spatial_xy(
    x,
    y,
    swap_xy=False,
    flip_x=False,
    flip_y=False,
):
    """Apply spatial coordinate transformations."""


    x = np.asarray(x, dtype=float).copy()
    y = np.asarray(y, dtype=float).copy()

    if swap_xy:
        x, y = y.copy(), x.copy()

    if flip_x:
        xmin = np.nanmin(x)
        xmax = np.nanmax(x)
        x = xmin + xmax - x

    if flip_y:
        ymin = np.nanmin(y)
        ymax = np.nanmax(y)
        y = ymin + ymax - y

    return x, y


In [ ]:
def plot_real_spatial_score(
    adata,

    sample,
    score_col,


    select_col="SM_source",
    select_value="D1_SM",

    sample_col="sample",

    x_col="x",
    y_col="y",


    normalize_01=True,


    vmin=0.30,
    vmax=1.00,


    cmap="seismic",


    show_other_cells=False,
    background_color="#D0D0D0",
    background_alpha=0.20,
    background_point_size=0.04,


    point_size=0.12,
    point_alpha=0.95,


    fig_width_mm=45,
    fig_height_mm=45,


    invert_y=True,
    swap_xy=False,
    flip_x=False,
    flip_y=False,


    show_title=True,
    title=None,
    colorbar_label=None,


    save=None,
    dpi=600,
    transparent=True,
):
    """Plot a selected score in tissue coordinates."""


    required_obs = [
        sample_col,
        select_col,
        score_col,
        x_col,
        y_col,
    ]

    missing = [
        col
        for col in required_obs
        if col not in adata.obs.columns
    ]

    if missing:
        raise KeyError(
            f"adata.obs is missing columns: {missing}"
        )


    sample_array = (
        adata.obs[sample_col]
        .astype(str)
        .to_numpy()
    )

    sample_mask = (
        sample_array == str(sample)
    )

    if sample_mask.sum() == 0:
        available = sorted(
            pd.unique(
                adata.obs[sample_col]
                .astype(str)
            )
        )

        raise ValueError(
            f"sample={sample!r} was not found.\n"
            f"Available samples:\n{available}"
        )


    select_array = (
        adata.obs[select_col]
        .astype(str)
        .to_numpy()
    )

    if isinstance(
        select_value,
        (list, tuple, set, np.ndarray, pd.Index)
    ):
        selected_mask = np.isin(
            select_array,
            [str(v) for v in select_value]
        )
    else:
        selected_mask = (
            select_array == str(select_value)
        )

    fg_mask = (
        sample_mask
        &
        selected_mask
    )


    score = pd.to_numeric(
        adata.obs[score_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    fg_mask &= np.isfinite(score)

    if fg_mask.sum() == 0:
        raise ValueError(
            f"sample={sample}, "
            f"{select_col}={select_value}, "
            f"score_col={score_col} "
            f"No valid cells."
        )


    x_all = pd.to_numeric(
        adata.obs[x_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_all = pd.to_numeric(
        adata.obs[y_col],
        errors="coerce"
    ).to_numpy(dtype=float)


    idx_sample = np.where(
        sample_mask
    )[0]

    x_sample = x_all[idx_sample]
    y_sample = y_all[idx_sample]

    x_sample, y_sample = _transform_spatial_xy(
        x_sample,
        y_sample,
        swap_xy=swap_xy,
        flip_x=flip_x,
        flip_y=flip_y,
    )


    x_plot = np.full(
        adata.n_obs,
        np.nan
    )

    y_plot = np.full(
        adata.n_obs,
        np.nan
    )

    x_plot[idx_sample] = x_sample
    y_plot[idx_sample] = y_sample


    plot_score = score.copy()

    if normalize_01:
        plot_score[fg_mask] = _minmax_01_spatial(
            score[fg_mask]
        )


    print("=" * 70)
    print(f"Sample      : {sample}")
    print(f"Selection   : {select_col} = {select_value}")
    print(f"Score       : {score_col}")
    print(f"N cells     : {fg_mask.sum():,}")
    print(f"Normalize01 : {normalize_01}")

    vals = plot_score[fg_mask]

    print(
        f"Plot score  : "
        f"{np.nanmin(vals):.4f} ~ "
        f"{np.nanmax(vals):.4f}"
    )

    print(
        f"Color range : "
        f"{vmin} ~ {vmax}"
    )


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )


    if show_other_cells:

        bg_mask = (
            sample_mask
            &
            (~fg_mask)
            &
            np.isfinite(x_plot)
            &
            np.isfinite(y_plot)
        )

        ax.scatter(
            x_plot[bg_mask],
            y_plot[bg_mask],

            s=background_point_size,
            c=background_color,

            alpha=background_alpha,

            linewidths=0,

            rasterized=True,
        )


    fg_indices = np.where(
        fg_mask
        &
        np.isfinite(x_plot)
        &
        np.isfinite(y_plot)
    )[0]

    order = np.argsort(
        plot_score[fg_indices]
    )

    fg_indices = fg_indices[
        order
    ]

    im = ax.scatter(
        x_plot[fg_indices],
        y_plot[fg_indices],

        s=point_size,

        c=plot_score[fg_indices],

        cmap=cmap,
        vmin=vmin,
        vmax=vmax,

        alpha=point_alpha,

        linewidths=0,

        rasterized=True,
    )


    if invert_y:
        ax.invert_yaxis()

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="box"
    )


    if show_title:

        if title is None:
            title = (
                f"{sample} | "
                f"{select_value} | "
                f"{score_col}"
            )

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.02,
    )

    if colorbar_label is None:

        if normalize_01:
            colorbar_label = (
                f"{score_col} (0–1)"
            )
        else:
            colorbar_label = (
                score_col
            )

    cbar.set_label(
        colorbar_label,
        fontsize=7,
        labelpad=2,
    )

    cbar.ax.tick_params(
        labelsize=7,
        width=0.5,
        length=2,
        pad=1,
    )

    cbar.outline.set_linewidth(
        0.5
    )


    if save is not None:

        save = Path(save)

        if save.parent != Path("."):
            save.parent.mkdir(
                parents=True,
                exist_ok=True
            )

        fig.savefig(
            save,
            dpi=dpi,
            bbox_inches="tight",
            pad_inches=0.02,
            transparent=transparent,
        )

        print(
            f"Saved:\n{save.resolve()}"
        )

    plt.show()

    return fig, ax


In [ ]:
print(
    sorted(
        adata.obs["sample"]
        .astype(str)
        .unique()
    )
)

SPATIAL_SAMPLE = (
    'WT_2'
)

print(
    "\nCurrent sample:",
    SPATIAL_SAMPLE
)


In [ ]:
fig_d1_s, ax_d1_s = plot_real_spatial_score(
    adata,

    sample=SPATIAL_SAMPLE,

    select_col="SM_source",
    select_value="D1_SM",

    score_col="SM_S_score",

    normalize_01=True,


    vmin=0,
    vmax=1.00,

    point_size=0.5,

    fig_width_mm=45,
    fig_height_mm=45,

    show_other_cells=False,

    invert_y=True,

    title=f"{SPATIAL_SAMPLE} | D1 | S-score",

    save=f"F:\BASSFISH_HD\Figure\Figure_3\Panel\{SPATIAL_SAMPLE}_D1_S_score_spatial.pdf",
)


In [ ]:
fig_d1_s, ax_d1_s = plot_real_spatial_score(
    adata,

    sample=SPATIAL_SAMPLE,

    select_col="SM_source",
    select_value="D1_SM",

    score_col="SM_M_score",

    normalize_01=True,


    vmin=0,
    vmax=1.00,

    point_size=0.5,

    fig_width_mm=45,
    fig_height_mm=45,

    show_other_cells=False,

    invert_y=True,

    title=f"{SPATIAL_SAMPLE} | D1 | M-score",

    save=f"F:\BASSFISH_HD\Figure\Figure_3\Panel\{SPATIAL_SAMPLE}_D1_M_score_spatial.pdf",
)


In [ ]:
fig_d1_s, ax_d1_s = plot_real_spatial_score(
    adata,

    sample=SPATIAL_SAMPLE,

    select_col="SM_source",
    select_value="D2_SM",

    score_col="SM_S_score",

    normalize_01=True,


    vmin=0,
    vmax=1.00,

    point_size=0.5,

    fig_width_mm=45,
    fig_height_mm=45,

    show_other_cells=False,

    invert_y=True,

    title=f"{SPATIAL_SAMPLE} | D2 | S-score",

    save=f"F:\BASSFISH_HD\Figure\Figure_3\Panel\{SPATIAL_SAMPLE}_D2_S_score_spatial.pdf",
)


In [ ]:
fig_d1_s, ax_d1_s = plot_real_spatial_score(
    adata,

    sample=SPATIAL_SAMPLE,

    select_col="SM_source",
    select_value="D2_SM",

    score_col="SM_M_score",

    normalize_01=True,


    vmin=0,
    vmax=1.00,

    point_size=0.5,

    fig_width_mm=45,
    fig_height_mm=45,

    show_other_cells=False,

    invert_y=True,

    title=f"{SPATIAL_SAMPLE} | D2 | M-score",

    save=f"F:\BASSFISH_HD\Figure\Figure_3\Panel\{SPATIAL_SAMPLE}_D2_M_score_spatial.pdf",
)


In [ ]:
DC1_COL = "SM_dc_component1"

DC1_OUTPUT_PDF = r"F:\BASSFISH_HD\Figure\Figure_3\Panel\DC1_umap.pdf"

DC1_FIG_WIDTH_MM = 52
DC1_FIG_HEIGHT_MM = 48

DC1_BG_POINT_SIZE = 0.05
DC1_FG_POINT_SIZE = 0.10

DC1_BG_ALPHA = 0.25
DC1_FG_ALPHA = 0.95

DC1_BG_COLOR = "#D0D0D0"

DC1_CMAP = "seismic"

DC1_VMIN = 0
DC1_VMAX = 1


umap = np.asarray(
    adata.obsm["X_umap"]
)

x = umap[:, 0]
y = umap[:, 1]


source = (
    adata.obs["SM_source"]
    .astype(str)
    .to_numpy()
)

dc1 = pd.to_numeric(
    adata.obs[DC1_COL],
    errors="coerce"
).to_numpy(dtype=float)


d1_mask = (
    (source == "D1_SM")
    &
    np.isfinite(dc1)
)

d2_mask = (
    (source == "D2_SM")
    &
    np.isfinite(dc1)
)

selected_mask = (
    d1_mask |
    d2_mask
)

background_mask = (
    ~selected_mask
)


def minmax01(x):

    x = np.asarray(
        x,
        dtype=float
    )

    xmin = np.nanmin(x)
    xmax = np.nanmax(x)

    if xmax == xmin:
        return np.full(
            len(x),
            0.5
        )

    return (
        x - xmin
    ) / (
        xmax - xmin
    )


dc1_plot = np.full(
    adata.n_obs,
    np.nan,
    dtype=float
)

dc1_plot[d1_mask] = minmax01(
    dc1[d1_mask]
)

dc1_plot[d2_mask] = minmax01(
    dc1[d2_mask]
)


print("D1 DC1:")
print(
    np.quantile(
        dc1_plot[d1_mask],
        [0, 0.05, 0.25, 0.5, 0.75, 0.95, 1]
    )
)

print("\nD2 DC1:")
print(
    np.quantile(
        dc1_plot[d2_mask],
        [0, 0.05, 0.25, 0.5, 0.75, 0.95, 1]
    )
)


mm_to_inch = 1 / 25.4

fig, ax = plt.subplots(
    figsize=(
        DC1_FIG_WIDTH_MM * mm_to_inch,
        DC1_FIG_HEIGHT_MM * mm_to_inch,
    )
)


ax.scatter(
    x[background_mask],
    y[background_mask],

    s=DC1_BG_POINT_SIZE,

    c=DC1_BG_COLOR,

    alpha=DC1_BG_ALPHA,

    linewidths=0,

    rasterized=True,
)


indices = np.where(
    selected_mask
)[0]

order = np.argsort(
    dc1_plot[indices]
)

indices = indices[order]


im = ax.scatter(
    x[indices],
    y[indices],

    s=DC1_FG_POINT_SIZE,

    c=dc1_plot[indices],

    cmap=DC1_CMAP,

    vmin=DC1_VMIN,
    vmax=DC1_VMAX,

    alpha=DC1_FG_ALPHA,

    linewidths=0,

    rasterized=True,
)


ax.set_xticks([])
ax.set_yticks([])

ax.set_xlabel("")
ax.set_ylabel("")

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_aspect(
    "equal",
    adjustable="datalim"
)


cbar = fig.colorbar(
    im,
    ax=ax,

    fraction=0.035,
    pad=0.02,
)

cbar.set_label(
    "DC1 (0–1)",
    fontsize=7,
    labelpad=2,
)

cbar.ax.tick_params(
    labelsize=7,
    width=0.5,
    length=2,
    pad=1,
)

cbar.outline.set_linewidth(
    0.5
)


fig.savefig(
    DC1_OUTPUT_PDF,

    dpi=600,

    bbox_inches="tight",
    pad_inches=0.02,

    transparent=True,
)

plt.show()


In [ ]:
def plot_spatial_dc(
    adata,

    sample,
    celltype="D1_SM",


    dc_col="SM_dc_component1",

    sample_col="sample",
    source_col="SM_source",

    x_col="x",
    y_col="y",


    normalize_01=True,


    cmap="seismic",
    vmin=0,
    vmax=1,


    point_size=0.15,
    point_alpha=0.95,


    fig_width_mm=45,
    fig_height_mm=45,


    swap_xy=False,
    flip_x=False,
    flip_y=False,
    invert_y=True,


    show_title=True,
    title=None,


    save=None,
    dpi=600,
    transparent=True,
):


    sample_array = (
        adata.obs[sample_col]
        .astype(str)
        .to_numpy()
    )

    source_array = (
        adata.obs[source_col]
        .astype(str)
        .to_numpy()
    )

    dc = pd.to_numeric(
        adata.obs[dc_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    mask = (
        (sample_array == str(sample))
        &
        (source_array == str(celltype))
        &
        np.isfinite(dc)
    )

    if mask.sum() == 0:
        raise ValueError(
            f"No cells found:\n"
            f"sample = {sample}\n"
            f"celltype = {celltype}\n"
            f"dc_col = {dc_col}"
        )


    x = pd.to_numeric(
        adata.obs.loc[mask, x_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    y = pd.to_numeric(
        adata.obs.loc[mask, y_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    score = dc[mask].copy()


    if swap_xy:
        x, y = y.copy(), x.copy()

    if flip_x:
        x = (
            np.nanmin(x)
            +
            np.nanmax(x)
            -
            x
        )

    if flip_y:
        y = (
            np.nanmin(y)
            +
            np.nanmax(y)
            -
            y
        )


    if normalize_01:

        smin = np.nanmin(score)
        smax = np.nanmax(score)

        if smax == smin:
            score_plot = np.full(
                len(score),
                0.5
            )

        else:
            score_plot = (
                score - smin
            ) / (
                smax - smin
            )

    else:

        score_plot = score


    print("=" * 60)
    print("sample:", sample)
    print("celltype:", celltype)
    print("DC:", dc_col)
    print("n cells:", mask.sum())

    print(
        "raw DC range:",
        np.nanmin(score),
        "~",
        np.nanmax(score)
    )

    print(
        "plot quantiles:",
        np.quantile(
            score_plot,
            [
                0,
                0.05,
                0.25,
                0.5,
                0.75,
                0.95,
                1,
            ]
        )
    )


    order = np.argsort(
        score_plot
    )

    x = x[order]
    y = y[order]
    score_plot = score_plot[order]


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )

    im = ax.scatter(
        x,
        y,

        s=point_size,

        c=score_plot,

        cmap=cmap,
        vmin=vmin,
        vmax=vmax,

        alpha=point_alpha,

        linewidths=0,

        rasterized=True,
    )


    if invert_y:
        ax.invert_yaxis()

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="box"
    )


    if show_title:

        if title is None:
            title = (
                f"{sample} | "
                f"{celltype} | DC1"
            )

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.02,
    )

    if normalize_01:
        cbar.set_label(
            "DC1 (0–1)",
            fontsize=7,
            labelpad=2,
        )
    else:
        cbar.set_label(
            "DC1",
            fontsize=7,
            labelpad=2,
        )

    cbar.ax.tick_params(
        labelsize=7,
        width=0.5,
        length=2,
        pad=1,
    )

    cbar.outline.set_linewidth(
        0.5
    )


    if save is not None:

        fig.savefig(
            save,

            dpi=dpi,

            bbox_inches="tight",
            pad_inches=0.02,

            transparent=transparent,
        )

        print("Saved:", save)

    plt.show()

    return fig, ax


In [ ]:
plot_spatial_dc(
    adata,

    sample="WT_2",
    celltype="D1_SM",

    dc_col="SM_dc_component1",

    normalize_01=True,

    cmap="seismic",
    vmin=0,
    vmax=1,

    point_size=1,

    fig_width_mm=45,
    fig_height_mm=45,

    save="F:\BASSFISH_HD\Figure\Figure_3\Panel\WT_2_D1_DC1_spatial.pdf",
)


In [ ]:
plot_spatial_dc(
    adata,

    sample="WT_2",
    celltype="D2_SM",

    dc_col="SM_dc_component1",

    normalize_01=True,

    cmap="seismic",
    vmin=0,
    vmax=1,

    point_size=1,

    fig_width_mm=45,
    fig_height_mm=45,

    save="F:\BASSFISH_HD\Figure\Figure_3\Panel\WT_2_D2_DC1_spatial.pdf",
)


In [ ]:
def plot_real_spatial_celltypes(
    adata,

    sample,
    celltypes,

    sample_col="sample",
    celltype_col="refined_cluster_4",

    x_col="x",
    y_col="y",

    color_dict=None,


    background_color="#D0D0D0",
    background_alpha=0.35,
    background_point_size=0.20,


    point_size=0.50,
    point_alpha=1.0,


    fig_width_mm=45,
    fig_height_mm=45,


    show_title=False,
    title=None,

    show_legend=False,


    save=None,
    dpi=600,
    transparent=True,
):

    if color_dict is None:
        raise ValueError("color_dict is required")

    if isinstance(celltypes, str):
        celltypes = [celltypes]


    sample_array = (
        adata.obs[sample_col]
        .astype(str)
        .to_numpy()
    )

    celltype_array = (
        adata.obs[celltype_col]
        .astype(str)
        .to_numpy()
    )

    sample_mask = (
        sample_array == str(sample)
    )

    if sample_mask.sum() == 0:
        raise ValueError(
            f"sample={sample!r} was not found"
        )


    x_plot = pd.to_numeric(
        adata.obs[x_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_plot = pd.to_numeric(
        adata.obs[y_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    valid_xy = (
        np.isfinite(x_plot)
        &
        np.isfinite(y_plot)
    )


    target_mask = (
        sample_mask
        &
        np.isin(
            celltype_array,
            [str(x) for x in celltypes]
        )
        &
        valid_xy
    )


    bg_mask = (
        sample_mask
        &
        (~np.isin(
            celltype_array,
            [str(x) for x in celltypes]
        ))
        &
        valid_xy
    )


    print("=" * 70)
    print("Sample:", sample)
    print("Total sample cells:", sample_mask.sum())
    print("Background cells:", bg_mask.sum())
    print("Target cells:", target_mask.sum())

    print("\nTarget cell types:")

    for ct in celltypes:

        if ct not in color_dict:
            raise KeyError(
                f"{ct!r} is missing from color_dict"
            )

        m = (
            sample_mask
            &
            (celltype_array == str(ct))
            &
            valid_xy
        )

        print(
            f"{ct}: {m.sum():,} | "
            f"{color_dict[ct]}"
        )


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            fig_width_mm * mm_to_inch,
            fig_height_mm * mm_to_inch,
        )
    )


    ax.scatter(
        x_plot[bg_mask],
        y_plot[bg_mask],

        s=background_point_size,
        c=background_color,

        alpha=background_alpha,

        linewidths=0,

        rasterized=True,
    )


    for ct in celltypes:

        fg_mask = (
            sample_mask
            &
            (celltype_array == str(ct))
            &
            valid_xy
        )

        if fg_mask.sum() == 0:
            continue

        ax.scatter(
            x_plot[fg_mask],
            y_plot[fg_mask],

            s=point_size,
            c=color_dict[ct],

            alpha=point_alpha,

            linewidths=0,

            rasterized=True,

            label=ct,
        )


    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect(
        "equal",
        adjustable="box"
    )


    if show_title:

        if title is None:
            title = str(sample)

        ax.set_title(
            title,
            fontsize=7,
            pad=2,
        )


    if show_legend:

        ax.legend(
            frameon=False,
            fontsize=7,

            loc="center left",
            bbox_to_anchor=(1.01, 0.5),

            markerscale=4,

            handletextpad=0.25,
            borderaxespad=0,
            labelspacing=0.35,
        )


    if save is not None:

        save = Path(save)

        if save.parent != Path("."):
            save.parent.mkdir(
                parents=True,
                exist_ok=True
            )

        fig.savefig(
            save,

            dpi=dpi,

            bbox_inches="tight",
            pad_inches=0.02,

            transparent=transparent,
        )

        print(
            f"\nSaved:\n{save.resolve()}"
        )

    plt.show()

    return fig, ax


In [ ]:
SPATIAL_SAMPLE = "WT_2"

celltypes_to_plot = [

    "Oligo",
    "OPC",
    "Astrocytes",
    "Mircrogila",
    "Vascular_endo",
    "Pericytes",
]

fig, ax = plot_real_spatial_celltypes(
    adata,

    sample=SPATIAL_SAMPLE,

    celltypes=celltypes_to_plot,

    color_dict=refined_cluster_4_cmap,


    background_color="#C2C1C1",
    background_alpha=0.6,
    background_point_size=0.50,


    point_size=0.8,
    point_alpha=0.8,


    fig_width_mm=45,
    fig_height_mm=45,

    show_title=False,
    show_legend=False,

    save=(
        rf"F:\BASSFISH_HD\Figure\Figure_3\Panel"
        rf"\{SPATIAL_SAMPLE}_other_type_selected_celltypes_spatial.pdf"
    ),

    dpi=600,
    transparent=True,
)


In [ ]:
SPATIAL_SAMPLE = "WT_2"

celltypes_to_plot = [

    "STR_chat",
    "STR_Rreb1_neuron",
    "D1H_Nxph4",
    "D1H_Cpne7",

    "Ventral_D1",
    "NAc_D1",
    "Dorsal_S_D1",
    "Dorsal_M_D1",

]

fig, ax = plot_real_spatial_celltypes(
    adata,

    sample=SPATIAL_SAMPLE,

    celltypes=celltypes_to_plot,

    color_dict=refined_cluster_4_cmap,


    background_color="#C2C1C1",
    background_alpha=0.6,
    background_point_size=0.50,


    point_size=0.8,
    point_alpha=0.8,


    fig_width_mm=45,
    fig_height_mm=45,

    show_title=False,
    show_legend=False,

    save=(
        rf"F:\BASSFISH_HD\Figure\Figure_3\Panel"
        rf"\{SPATIAL_SAMPLE}_d1_selected_celltypes_spatial.pdf"
    ),

    dpi=600,
    transparent=True,
)


In [ ]:
SPATIAL_SAMPLE = "WT_2"

celltypes_to_plot = [

    "NAc_D2",
    "Dorsal_S_D2",

    "Dorsal_M_D2",
    "Ventral_D2",

]

fig, ax = plot_real_spatial_celltypes(
    adata,

    sample=SPATIAL_SAMPLE,

    celltypes=celltypes_to_plot,

    color_dict=refined_cluster_4_cmap,


    background_color="#C2C1C1",
    background_alpha=0.6,
    background_point_size=0.50,


    point_size=0.8,
    point_alpha=0.8,


    fig_width_mm=45,
    fig_height_mm=45,

    show_title=False,
    show_legend=False,

    save=(
        rf"F:\BASSFISH_HD\Figure\Figure_3\Panel"
        rf"\{SPATIAL_SAMPLE}_d2_selected_celltypes_spatial.pdf"
    ),

    dpi=600,
    transparent=True,
)


## Panel G | DC1 correlations with S-score and M-score


In [ ]:
# Imports and settings.
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.backends.backend_agg import FigureCanvasAgg
from scipy.stats import pearsonr, spearmanr


In [ ]:
H5AD_PATH = Path(
    r"F:\BASSFISH_HD\Source_data\integrated_str_final_with_D1_D2_DC.h5ad"
)


OUTPUT_DIR = Path(
    r"F:\BASSFISH_HD\Figure\Figure_3\Panel\score_DC1_correlation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SOURCE_COL = "SM_source"
S_SCORE_COL = "SM_S_score"
M_SCORE_COL = "SM_M_score"
DC1_RAW_COL = "SM_dc_component1"
SAMPLE_COL = "sample"


SOURCE_VALUES = [
    "D1_SM",
    "D2_SM",
]


SAMPLE = None


FIG_WIDTH_MM = 60
FIG_HEIGHT_MM = 40


POINT_COLOR = "#1F77B4"
POINT_SIZE = 0.70


REGRESSION_COLOR = "#000000"
REGRESSION_WIDTH = 0.8


DPI = 600


SHOW_REGRESSION = True
SHOW_STATS = True


In [ ]:
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,
    "ytick.major.size": 2,
    "savefig.transparent": False,
})


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)

print(adata)

required_cols = [
    SOURCE_COL,
    S_SCORE_COL,
    M_SCORE_COL,
    DC1_RAW_COL,
]

if SAMPLE is not None:
    required_cols.append(SAMPLE_COL)

missing = [
    col
    for col in required_cols
    if col not in adata.obs.columns
]

if missing:
    raise KeyError(
        f"adata.obs is missing columns: {missing}"
    )


In [ ]:
cols = [
    SOURCE_COL,
    S_SCORE_COL,
    M_SCORE_COL,
    DC1_RAW_COL,
]

if SAMPLE_COL in adata.obs.columns:
    cols.append(SAMPLE_COL)

df = adata.obs[cols].copy()

df[SOURCE_COL] = (
    df[SOURCE_COL]
    .astype(str)
)

for col in [
    S_SCORE_COL,
    M_SCORE_COL,
    DC1_RAW_COL,
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce",
    )


df = df[
    df[SOURCE_COL].isin(SOURCE_VALUES)
].copy()


if SAMPLE is not None:
    df = df[
        df[SAMPLE_COL].astype(str)
        == str(SAMPLE)
    ].copy()


DC1_NORM_COL = "DC1_0_100"

df[DC1_NORM_COL] = np.nan

for source in SOURCE_VALUES:

    mask = (
        (df[SOURCE_COL] == source)
        & np.isfinite(df[DC1_RAW_COL])
    )

    vals = df.loc[
        mask,
        DC1_RAW_COL,
    ].to_numpy(dtype=float)

    if len(vals) == 0:
        print(
            f"Warning: {source} has no valid DC1 values"
        )
        continue

    vmin = np.nanmin(vals)
    vmax = np.nanmax(vals)

    if vmax == vmin:
        norm = np.full(
            len(vals),
            50.0,
        )
    else:
        norm = (
            (vals - vmin)
            / (vmax - vmin)
            * 100.0
        )

    df.loc[
        mask,
        DC1_NORM_COL,
    ] = norm

    print(
        f"{source}: "
        f"n={mask.sum():,}, "
        f"raw DC1={vmin:.6g} ~ {vmax:.6g} "
        f"-> 0 ~ 100"
    )

print(
    f"\nSelected cells: {len(df):,}"
)


In [ ]:
def calculate_correlation(x, y):

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 3:
        raise ValueError(
            "Fewer than 3 valid observations"
        )

    pearson_r, pearson_p = pearsonr(
        x,
        y,
    )

    spearman_rho, spearman_p = spearmanr(
        x,
        y,
    )

    return {
        "n": len(x),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
    }


def format_pvalue(p):

    if p < 1e-4:
        return "<1e-4"

    return f"{p:.4f}"


In [ ]:
def render_scatter_rgb(
    x,
    y,
    xlim,
    ylim,
    width_px,
    height_px,
    dpi=600,
    point_color="#9FD8E2",
    point_size=0.70,
):
    """Render scatter points as an RGB image with Agg."""


    fig = plt.Figure(
        figsize=(
            width_px / dpi,
            height_px / dpi,
        ),
        dpi=dpi,
        facecolor="white",
    )

    canvas = FigureCanvasAgg(fig)

    ax = fig.add_axes([
        0,
        0,
        1,
        1,
    ])

    ax.set_facecolor("white")

    ax.scatter(
        x,
        y,
        s=point_size,
        c=point_color,
        alpha=1.0,
        edgecolors="none",
        linewidths=0,
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.axis("off")

    canvas.draw()

    rgba = np.asarray(
        canvas.buffer_rgba()
    )


    rgb = rgba[
        :, :, :3
    ].copy()

    plt.close(fig)

    return rgb


In [ ]:
def plot_dc1_score_correlation(
    df,
    score_col,
    score_label,
    output_name,
):


    plot_df = df[
        [
            DC1_NORM_COL,
            score_col,
        ]
    ].copy()

    valid = (
        np.isfinite(
            plot_df[DC1_NORM_COL]
            .to_numpy(dtype=float)
        )
        &
        np.isfinite(
            plot_df[score_col]
            .to_numpy(dtype=float)
        )
    )

    plot_df = plot_df.loc[
        valid
    ].copy()

    x = plot_df[
        DC1_NORM_COL
    ].to_numpy(dtype=float)

    y = plot_df[
        score_col
    ].to_numpy(dtype=float)


    stat = calculate_correlation(
        x,
        y,
    )

    print("=" * 70)
    print(
        f"DC1 vs {score_label}"
    )
    print(
        f"N = {stat['n']:,}"
    )
    print(
        f"Pearson r = "
        f"{stat['pearson_r']:.6f}"
    )
    print(
        f"Pearson p = "
        f"{stat['pearson_p']:.3e}"
    )
    print(
        f"Spearman rho = "
        f"{stat['spearman_rho']:.6f}"
    )
    print(
        f"Spearman p = "
        f"{stat['spearman_p']:.3e}"
    )


    xlim = (
        0,
        100,
    )

    y_min = np.nanmin(y)
    y_max = np.nanmax(y)

    y_range = (
        y_max - y_min
    )

    if y_range == 0:
        y_range = 1.0

    ylim = (
        y_min - y_range * 0.03,
        y_max + y_range * 0.03,
    )


    mm_to_inch = 1 / 25.4

    fig, ax = plt.subplots(
        figsize=(
            FIG_WIDTH_MM * mm_to_inch,
            FIG_HEIGHT_MM * mm_to_inch,
        ),
        dpi=DPI,
        facecolor="white",
    )

    ax.set_facecolor("white")


    fig.subplots_adjust(
        left=0.20,
        right=0.96,
        bottom=0.18,
        top=0.96,
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


    fig.canvas.draw()

    bbox = ax.get_window_extent()

    axes_width_px = max(
        1,
        int(round(bbox.width)),
    )

    axes_height_px = max(
        1,
        int(round(bbox.height)),
    )

    print(
        f"Scatter RGB size: "
        f"{axes_width_px} × {axes_height_px} px"
    )


    scatter_rgb = render_scatter_rgb(
        x=x,
        y=y,
        xlim=xlim,
        ylim=ylim,
        width_px=axes_width_px,
        height_px=axes_height_px,
        dpi=DPI,
        point_color=POINT_COLOR,
        point_size=POINT_SIZE,
    )


    ax.imshow(
        scatter_rgb,
        extent=(
            xlim[0],
            xlim[1],
            ylim[0],
            ylim[1],
        ),
        origin="upper",
        interpolation="none",
        aspect="auto",
        zorder=1,
    )


    if SHOW_REGRESSION:

        slope, intercept = np.polyfit(
            x,
            y,
            1,
        )

        xx = np.linspace(
            0,
            100,
            200,
        )

        yy = (
            slope * xx
            + intercept
        )

        ax.plot(
            xx,
            yy,
            color=REGRESSION_COLOR,
            linewidth=REGRESSION_WIDTH,
            zorder=10,
        )


    if SHOW_STATS:

        stat_text = (
            f"n = {stat['n']:,}\n"
            f"Pearson r = {stat['pearson_r']:.3f}\n"
            f"P = {format_pvalue(stat['pearson_p'])}\n"
            f"Spearman ρ = {stat['spearman_rho']:.3f}"
        )

        ax.text(
            0.04,
            0.96,
            stat_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7,
            zorder=20,
        )


    ax.set_xlabel(
        "DC1 (0–100)"
    )

    ax.set_ylabel(
        score_label
    )

    ax.set_xlim(
        0,
        100,
    )

    ax.set_ylim(ylim)

    ax.set_xticks([
        0,
        20,
        40,
        60,
        80,
        100,
    ])

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)

    ax.tick_params(
        direction="out",
        width=0.5,
        length=2,
        pad=1,
    )


    pdf_path = (
        OUTPUT_DIR
        / f"{output_name}.pdf"
    )

    fig.savefig(
        pdf_path,
        format="pdf",
        dpi=DPI,
        transparent=False,
        facecolor="white",
        edgecolor="white",
        bbox_inches=None,
        pad_inches=0,
    )

    print(
        f"Saved PDF: {pdf_path}"
    )

    plt.show()

    return stat


In [ ]:
stats_s = plot_dc1_score_correlation(
    df,
    score_col=S_SCORE_COL,
    score_label="S-score",
    output_name="DC1_vs_S_score",
)


In [ ]:
stats_m = plot_dc1_score_correlation(
    df,
    score_col=M_SCORE_COL,
    score_label="M-score",
    output_name="DC1_vs_M_score",
)


In [ ]:
stats_df = pd.DataFrame([
    {
        "comparison": "DC1_vs_S_score",
        **stats_s,
    },
    {
        "comparison": "DC1_vs_M_score",
        **stats_m,
    },
])

stats_csv = (
    OUTPUT_DIR
    / "DC1_score_correlation_stats.csv"
)

stats_df.to_csv(
    stats_csv,
    index=False,
)

print(
    f"Saved: {stats_csv}"
)

display(stats_df)


## Panel I | Single-gene tissue maps

The spatial-map example uses `Tshz1`, `Sgk1`, the dorsal region, and the `raw` layer. Exact panel I calls are not included. Panel genes are Nefm/Pde1a/Oprm1/Wfs1 (dorsal striatum), Zfhx3/Spock3/Htr4/Id4 (NAcc), and Nefm/Pde1a (OT). Check the input data, `GENES`, `REGIONS`, color limits, and `COLORBAR_LABEL` for the intended plot. The example colorbar says Spatial z-score while `LAYER='raw'`.


In [ ]:
# Imports and settings.
from pathlib import Path
import re
import pandas as pd
import scanpy as sc


H5AD_FILE = Path(r"F:\BASSFISH_HD\Source_data\integrated_5_str_with_sample_level_per_cell_layers.h5ad")

OUT_DIR = Path(r"F:\BASSFISH_HD\Figure\Figure_3\Panel\show_single_gene")


GENES = ["Tshz1", "Sgk1"]


REGIONS = ["d"]


LAYER = "raw"
SAMPLE_COL = "sample"
CLUSTER_COL = "refined_cluster_6_merged"
REGION_COL = "region_str_manual"
X_COL = "x"
Y_COL = "y"


CLUSTER_SETS = {
    "D1": "MSN_D1",
    "D2": "MSN_D2",
    "D1_plus_D2": ["MSN_D1", "MSN_D2"],
}


FLIP_ORIGINAL_SAMPLE_COORDINATES = True
SAMPLES_FLIP_X = ["Homo_3", "WT_5"]
SAMPLES_FLIP_Y = ["Homo_4", "WT_2", "WT_3", "WT_4"]


BIN_SIZE = 124
MIN_CELLS_PER_BIN = 1
SMOOTH_SIGMA = 1
SMOOTH_EXPAND = True
SMOOTH_WEIGHT_THRESHOLD = 0.05


DEFAULT_COLOR_LIMIT = 3
PER_GENE_COLOR_LIMIT = {


}
CMAP = "RdBu_r"


NCOLS = 4
PANEL_WIDTH_MM = 12
SHOW_SAMPLE_TITLE = True
SHOW_CELL_NUMBER = True
SHOW_SCALE_BAR = True
SCALE_BAR_UM = 500
UM_PER_COORDINATE = 0.16
SCALE_BAR_EACH_PANEL = True
SHOW_COLORBAR = True
COLORBAR_WIDTH_MM = 1.2
COLORBAR_HEIGHT_MM = 18
COLORBAR_LABEL = "Spatial z-score"
SAVE_DPI = 600


SHOW_FIGURES = False


In [ ]:
if not H5AD_FILE.is_file():
    raise FileNotFoundError(f"h5ad file not found: {H5AD_FILE}")

adata = sc.read_h5ad(H5AD_FILE)
print(adata)

for col in [SAMPLE_COL, CLUSTER_COL, REGION_COL, X_COL, Y_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing obs[{col!r}]; available columns: {list(adata.obs.columns)}")

if LAYER not in adata.layers:
    raise KeyError(f"Missing layers[{LAYER!r}]; available layers: {list(adata.layers.keys())}")

if FLIP_ORIGINAL_SAMPLE_COORDINATES:
    adata.obs[X_COL + "_original"] = adata.obs[X_COL].copy()
    adata.obs[Y_COL + "_original"] = adata.obs[Y_COL].copy()

    sample_id = adata.obs[SAMPLE_COL].astype(str)
    adata.obs.loc[sample_id.isin(SAMPLES_FLIP_X), X_COL] *= -1
    adata.obs.loc[sample_id.isin(SAMPLES_FLIP_Y), Y_COL] *= -1
    print("Applied the configured sample-specific x/y coordinate flips in memory.")

clusters_available = set(adata.obs[CLUSTER_COL].astype(str).unique())
for required_cluster in ["MSN_D1", "MSN_D2"]:
    if required_cluster not in clusters_available:
        raise ValueError(
            f"obs[{CLUSTER_COL!r}] does not contain {required_cluster}.\n"
            f"Available categories: {sorted(clusters_available)}"
        )

if GENES is None:
    genes_to_plot = adata.var_names.astype(str).tolist()
else:
    genes_to_plot = list(dict.fromkeys(str(g) for g in GENES))

missing_genes = [g for g in genes_to_plot if g not in adata.var_names]
if missing_genes:
    raise ValueError(f"Genes missing from adata.var_names: {missing_genes}")

if not genes_to_plot:
    raise ValueError("GENES is empty.")

for region in REGIONS:
    if str(region) not in set(adata.obs[REGION_COL].astype(str)):
        raise ValueError(f"Region {region!r} is absent from obs[{REGION_COL!r}]")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plotting {len(genes_to_plot)} genes x {len(REGIONS)} regions x 3 cell-type combinations.")


In [ ]:
# Imports and settings.
import math
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.transforms import blended_transform_factory
from scipy import sparse
from scipy.ndimage import gaussian_filter


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 7,

    "axes.titlesize": 7,
    "axes.labelsize": 7,

    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    "legend.fontsize": 7,
    "legend.title_fontsize": 7,

    "figure.titlesize": 7,


    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.unicode_minus": False,
})


def _to_list(value):
    """Convert one value or a sequence to a list."""


    if value is None:
        return None

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return list(value)

    return [value]


def _add_scale_bar(
    ax,
    length_um=500,
    um_per_coordinate=0.16,
    color="black",
    linewidth=0.8,
    fontsize=7,
    x_pad_fraction=0.06,
    y_position_fraction=0.07,
    text_gap_fraction=0.025,
):
    """Add a scale bar to the spatial plot."""


    if length_um is None:
        return

    if length_um <= 0:
        raise ValueError("length_um must be greater than 0")

    if um_per_coordinate <= 0:
        raise ValueError("um_per_coordinate must be greater than 0")


    length_coordinate = length_um / um_per_coordinate

    x_min, x_max = sorted(ax.get_xlim())
    x_range = x_max - x_min

    if x_range <= 0:
        return

    if length_coordinate >= x_range:
        raise ValueError(
            f"Scale bar too long: {length_um} micrometers corresponds to "
            f"{length_coordinate:.1f} coordinate units; "
            f"the subplot width is only {x_range:.1f} coordinate units."
        )

    x_right = x_max - x_pad_fraction * x_range
    x_left = x_right - length_coordinate
    x_center = (x_left + x_right) / 2


    transform = blended_transform_factory(
        ax.transData,
        ax.transAxes,
    )


    ax.plot(
        [x_left, x_right],
        [y_position_fraction, y_position_fraction],
        transform=transform,
        color=color,
        linewidth=linewidth,
        solid_capstyle="butt",
        clip_on=False,
        zorder=100,
    )


    ax.text(
        x_center,
        y_position_fraction + text_gap_fraction,
        f"{length_um:g} µm",
        transform=transform,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontfamily="Arial",
        fontweight="normal",
        color=color,
        clip_on=False,
        zorder=100,
    )


def plot_spatial_heatmap_all_samples(
    adata,
    gene,
    clusters=None,
    regions=None,


    layer="spatial_z_cell",
    sample_col="sample",
    cluster_col="refined_cluster_6_merged",
    region_col="region_str_manual",
    x_col="x",
    y_col="y",


    bin_size=50,
    min_cells_per_bin=1,
    smooth_sigma=1,
    smooth_expand=True,
    smooth_weight_threshold=0.05,


    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    percentile=99,


    ncols=4,
    panel_width_mm=12,
    invert_y=True,
    use_full_sample_extent=True,
    wspace=0.12,
    hspace=0.30,


    title="",
    show_sample_title=True,
    show_cell_number=True,


    show_colorbar=True,
    colorbar_label=None,
    colorbar_width_mm=1.2,
    colorbar_height_mm=18,
    colorbar_gap_mm=2,


    show_scale_bar=True,
    scale_bar_um=500,
    um_per_coordinate=0.16,
    scale_bar_color="black",
    scale_bar_linewidth=0.8,
    scale_bar_each_panel=True,


    save_path=None,
    save_dpi=600,
    show=True,
):
    """Plot sample-wise spatial expression heatmaps."""


    font_size = 7
    mm_to_inch = 1 / 25.4


    if gene not in adata.var_names:
        raise ValueError(
            f"Gene {gene!r} is missing from adata.var_names"
        )

    if layer not in adata.layers:
        raise ValueError(
            f"Layer {layer!r} is missing from adata.layers"
        )

    required_cols = [
        sample_col,
        cluster_col,
        region_col,
        x_col,
        y_col,
    ]

    for col in required_cols:
        if col not in adata.obs.columns:
            raise ValueError(
                f"Column {col!r} is missing from adata.obs"
            )

    if bin_size <= 0:
        raise ValueError("bin_size must be greater than 0")

    if min_cells_per_bin < 1:
        raise ValueError(
            "min_cells_per_bin must be at least 1"
        )

    if smooth_sigma < 0:
        raise ValueError(
            "smooth_sigma must be nonnegative"
        )

    if ncols < 1:
        raise ValueError("ncols must be at least 1")

    if panel_width_mm <= 0:
        raise ValueError(
            "panel_width_mm must be greater than 0"
        )


    clusters_list = _to_list(clusters)

    if clusters_list is None:
        cluster_mask = np.ones(
            adata.n_obs,
            dtype=bool,
        )
        cluster_title = "all"

    else:
        clusters_str = [
            str(value)
            for value in clusters_list
        ]

        cluster_mask = (
            adata.obs[cluster_col]
            .astype(str)
            .isin(clusters_str)
            .to_numpy()
        )

        cluster_title = ", ".join(clusters_str)


    regions_list = _to_list(regions)

    if regions_list is None:
        region_mask = np.ones(
            adata.n_obs,
            dtype=bool,
        )
        region_title = "all"

    else:
        regions_str = [
            str(value)
            for value in regions_list
        ]

        region_mask = (
            adata.obs[region_col]
            .astype(str)
            .isin(regions_str)
            .to_numpy()
        )

        region_title = ", ".join(regions_str)

    selected_mask = cluster_mask & region_mask

    if selected_mask.sum() == 0:
        raise ValueError(
            "No cells remain after filtering.\n"
            f"{cluster_col}={clusters}\n"
            f"{region_col}={regions}"
        )


    values_all = adata[:, gene].layers[layer]

    if sparse.issparse(values_all):
        values_all = values_all.toarray()

    values_all = (
        np.asarray(values_all)
        .ravel()
        .astype(float)
    )

    x_all = adata.obs[x_col].to_numpy(
        dtype=float
    )

    y_all = adata.obs[y_col].to_numpy(
        dtype=float
    )

    sample_values = (
        adata.obs[sample_col]
        .astype(str)
        .to_numpy()
    )

    samples = (
        adata.obs.loc[selected_mask, sample_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    samples = sorted(samples)

    if len(samples) == 0:
        raise ValueError(
            "No samples remain for plotting after filtering"
        )


    results = []
    all_heatmap_values = []

    for sample_name in samples:

        sample_mask = (
            sample_values == sample_name
        )

        plot_mask = (
            selected_mask & sample_mask
        )

        x = x_all[plot_mask]
        y = y_all[plot_mask]
        z = values_all[plot_mask]

        valid = (
            np.isfinite(x)
            & np.isfinite(y)
            & np.isfinite(z)
        )

        x = x[valid]
        y = y[valid]
        z = z[valid]

        if len(x) == 0:
            results.append(None)
            continue


        if use_full_sample_extent:

            x_reference = x_all[sample_mask]
            y_reference = y_all[sample_mask]

            reference_valid = (
                np.isfinite(x_reference)
                & np.isfinite(y_reference)
            )

            x_reference = x_reference[
                reference_valid
            ]

            y_reference = y_reference[
                reference_valid
            ]

        else:
            x_reference = x
            y_reference = y

        if len(x_reference) == 0:
            results.append(None)
            continue

        x_min = np.nanmin(x_reference)
        x_max = np.nanmax(x_reference)
        y_min = np.nanmin(y_reference)
        y_max = np.nanmax(y_reference)

        if x_max == x_min:
            x_max = x_min + bin_size

        if y_max == y_min:
            y_max = y_min + bin_size


        x_start = (
            np.floor(x_min / bin_size)
            * bin_size
        )

        x_end = (
            np.ceil(x_max / bin_size)
            * bin_size
        )

        y_start = (
            np.floor(y_min / bin_size)
            * bin_size
        )

        y_end = (
            np.ceil(y_max / bin_size)
            * bin_size
        )

        x_edges = np.arange(
            x_start,
            x_end + bin_size,
            bin_size,
        )

        y_edges = np.arange(
            y_start,
            y_end + bin_size,
            bin_size,
        )

        if len(x_edges) < 2:
            x_edges = np.array([
                x_start,
                x_start + bin_size,
            ])

        if len(y_edges) < 2:
            y_edges = np.array([
                y_start,
                y_start + bin_size,
            ])


        count_map, _, _ = np.histogram2d(
            y,
            x,
            bins=[y_edges, x_edges],
        )


        sum_map, _, _ = np.histogram2d(
            y,
            x,
            bins=[y_edges, x_edges],
            weights=z,
        )


        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            mean_map = sum_map / count_map

        occupied_mask = (
            count_map >= min_cells_per_bin
        )

        mean_map[~occupied_mask] = np.nan


        if smooth_sigma > 0:

            finite_mask = np.isfinite(
                mean_map
            )

            value_filled = np.nan_to_num(
                mean_map,
                nan=0.0,
            )

            weight_map = finite_mask.astype(
                float
            )

            smooth_value = gaussian_filter(
                value_filled * weight_map,
                sigma=smooth_sigma,
                mode="constant",
                cval=0,
            )

            smooth_weight = gaussian_filter(
                weight_map,
                sigma=smooth_sigma,
                mode="constant",
                cval=0,
            )

            with np.errstate(
                divide="ignore",
                invalid="ignore",
            ):
                smooth_map = (
                    smooth_value / smooth_weight
                )

            if smooth_expand:

                smooth_map[
                    smooth_weight
                    < smooth_weight_threshold
                ] = np.nan

            else:

                smooth_map[
                    ~occupied_mask
                ] = np.nan

            mean_map = smooth_map

        valid_heatmap = mean_map[
            np.isfinite(mean_map)
        ]

        if valid_heatmap.size > 0:
            all_heatmap_values.append(
                valid_heatmap
            )

        results.append({
            "sample": sample_name,
            "map": mean_map,
            "count_map": count_map,
            "x_edges": x_edges,
            "y_edges": y_edges,
            "n_cells": int(plot_mask.sum()),
        })

    if len(all_heatmap_values) == 0:
        raise ValueError(
            "No samples have valid heatmaps.\n"
            "Increase bin_size, decrease min_cells_per_bin, "
            "or check the filtering criteria."
        )

    all_heatmap_values = np.concatenate(
        all_heatmap_values
    )


    if vmin is None and vmax is None:

        vmax_plot = np.nanpercentile(
            np.abs(all_heatmap_values),
            percentile,
        )

        if (
            not np.isfinite(vmax_plot)
            or vmax_plot == 0
        ):
            vmax_plot = 1.0

        vmin_plot = -vmax_plot

    elif vmin is not None and vmax is not None:

        vmin_plot = float(vmin)
        vmax_plot = float(vmax)

    elif vmin is None:

        vmax_plot = float(vmax)
        vmin_plot = -abs(vmax_plot)

    else:

        vmin_plot = float(vmin)
        vmax_plot = abs(vmin_plot)

    if vmin_plot >= vmax_plot:
        raise ValueError(
            f"Invalid color limits: "
            f"vmin={vmin_plot}, "
            f"vmax={vmax_plot}"
        )


    nrows = math.ceil(
        len(samples) / ncols
    )

    panel_width_inch = (
        panel_width_mm * mm_to_inch
    )


    aspect_ratios = []

    for result in results:

        if result is None:
            continue

        x_range = (
            result["x_edges"][-1]
            - result["x_edges"][0]
        )

        y_range = (
            result["y_edges"][-1]
            - result["y_edges"][0]
        )

        if x_range > 0 and y_range > 0:
            aspect_ratios.append(
                y_range / x_range
            )

    max_aspect_ratio = (
        max(aspect_ratios)
        if len(aspect_ratios) > 0
        else 1.0
    )


    plot_height_inch = (
        panel_width_inch
        * max_aspect_ratio
    )


    title_extra_mm = (
        4 if show_sample_title else 1
    )

    row_slot_height_inch = (
        plot_height_inch
        + title_extra_mm * mm_to_inch
    )


    left = 0.03
    right = 0.84 if show_colorbar else 0.98
    bottom = 0.03

    if title not in (None, ""):
        top = 0.91
    else:
        top = 0.97


    figure_width_inch = (
        panel_width_inch
        * (
            ncols
            + (ncols - 1) * wspace
        )
        / (right - left)
    )


    figure_height_inch = (
        row_slot_height_inch
        * (
            nrows
            + (nrows - 1) * hspace
        )
        / (top - bottom)
    )

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(
            figure_width_inch,
            figure_height_inch,
        ),
        squeeze=False,
        constrained_layout=False,
    )

    fig.subplots_adjust(
        left=left,
        right=right,
        bottom=bottom,
        top=top,
        wspace=wspace,
        hspace=hspace,
    )

    mesh = None


    for panel_index, (ax, result) in enumerate(
        zip(axes.flat, results)
    ):

        if result is None:
            ax.axis("off")
            continue

        heatmap_masked = np.ma.masked_invalid(
            result["map"]
        )

        mesh = ax.pcolormesh(
            result["x_edges"],
            result["y_edges"],
            heatmap_masked,
            cmap=cmap,
            vmin=vmin_plot,
            vmax=vmax_plot,
            shading="flat",
            edgecolors="none",
            linewidth=0,
            antialiased=False,


            rasterized=True,
        )

        ax.set_xlim(
            result["x_edges"][0],
            result["x_edges"][-1],
        )

        ax.set_ylim(
            result["y_edges"][0],
            result["y_edges"][-1],
        )

        ax.set_aspect(
            "equal",
            adjustable="box",
        )

        if invert_y:
            ax.invert_yaxis()


        if show_sample_title:
            sample_title = result["sample"]

            ax.set_title(
                sample_title,
                fontsize=font_size,
                fontfamily="Arial",
                fontweight="normal",
                pad=1.5,
            )

        ax.axis("off")


        if show_scale_bar:

            if (
                scale_bar_each_panel
                or panel_index == 0
            ):

                _add_scale_bar(
                    ax=ax,
                    length_um=scale_bar_um,
                    um_per_coordinate=(
                        um_per_coordinate
                    ),
                    color=scale_bar_color,
                    linewidth=(
                        scale_bar_linewidth
                    ),
                    fontsize=font_size,
                )


    for ax in axes.flat[len(samples):]:
        ax.axis("off")


    if show_colorbar and mesh is not None:

        colorbar_width_inch = (
            colorbar_width_mm
            * mm_to_inch
        )

        colorbar_height_inch = (
            colorbar_height_mm
            * mm_to_inch
        )

        colorbar_gap_inch = (
            colorbar_gap_mm
            * mm_to_inch
        )

        colorbar_left_inch = (
            figure_width_inch * right
            + colorbar_gap_inch
        )

        colorbar_bottom_inch = (
            figure_height_inch
            - colorbar_height_inch
        ) / 2

        cax = fig.add_axes([
            colorbar_left_inch
            / figure_width_inch,

            colorbar_bottom_inch
            / figure_height_inch,

            colorbar_width_inch
            / figure_width_inch,

            colorbar_height_inch
            / figure_height_inch,
        ])

        cbar = fig.colorbar(
            mesh,
            cax=cax,
        )

        if colorbar_label is None:
            colorbar_label = gene

        cbar.set_label(
            colorbar_label,
            rotation=90,
            labelpad=4,
            fontsize=font_size,
            fontfamily="Arial",
            fontweight="normal",
        )

        cbar.ax.tick_params(
            labelsize=font_size,
            length=2,
            width=0.5,
        )

        for tick_label in (
            cbar.ax.get_yticklabels()
        ):
            tick_label.set_fontfamily(
                "Arial"
            )
            tick_label.set_fontsize(
                font_size
            )

        cbar.outline.set_linewidth(0.5)


    if title is None:
        title = (
            f"{gene} | "
            f"{cluster_col}: {cluster_title} | "
            f"{region_col}: {region_title}"
        )

    if title != "":
        fig.suptitle(
            title,
            fontsize=font_size,
            fontfamily="Arial",
            fontweight="normal",
            y=0.98,
        )


    if save_path is not None:

        save_path = Path(save_path)

        if save_path.suffix.lower() != ".pdf":
            save_path = (
                save_path.with_suffix(".pdf")
            )

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            save_path,
            format="pdf",


            dpi=save_dpi,


            bbox_inches=None,
            pad_inches=0,

            facecolor="white",
            transparent=False,
        )

        print(
            f"PDF saved: "
            f"{save_path.resolve()}"
        )

        print(
            f"PDF page dimensions: "
            f"{figure_width_inch * 25.4:.2f} × "
            f"{figure_height_inch * 25.4:.2f} mm"
        )

        print(
            f"Target width of each spatial plot: "
            f"{panel_width_mm:.2f} mm"
        )


    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes, results


In [ ]:
def safe_filename(value):
    return re.sub(r'[<>:"/\\|?*\s]+', "_", str(value)).strip("._") or "unnamed"


manifest = []
for region in REGIONS:
    for gene in genes_to_plot:
        limit = float(PER_GENE_COLOR_LIMIT.get(gene, DEFAULT_COLOR_LIMIT))
        if limit <= 0:
            raise ValueError(f"{gene}: color limit must be greater than 0")

        for mode, cluster_selection in CLUSTER_SETS.items():
            output_pdf = (
                OUT_DIR
                / f"{safe_filename(gene)}__{safe_filename(region)}__{mode}__{safe_filename(LAYER)}.pdf"
            )
            print(f"\n>>> {gene} | region={region} | {mode}")

            fig, axes, sample_results = plot_spatial_heatmap_all_samples(
                adata=adata,
                gene=gene,
                layer=LAYER,
                clusters=cluster_selection,
                regions=region,
                sample_col=SAMPLE_COL,
                cluster_col=CLUSTER_COL,
                region_col=REGION_COL,
                x_col=X_COL,
                y_col=Y_COL,
                bin_size=BIN_SIZE,
                min_cells_per_bin=MIN_CELLS_PER_BIN,
                smooth_sigma=SMOOTH_SIGMA,
                smooth_expand=SMOOTH_EXPAND,
                smooth_weight_threshold=SMOOTH_WEIGHT_THRESHOLD,
                cmap=CMAP,
                vmin=-limit,
                vmax=limit,
                ncols=NCOLS,
                panel_width_mm=PANEL_WIDTH_MM,
                invert_y=True,
                use_full_sample_extent=True,
                show_sample_title=SHOW_SAMPLE_TITLE,
                show_cell_number=SHOW_CELL_NUMBER,
                show_scale_bar=SHOW_SCALE_BAR,
                scale_bar_um=SCALE_BAR_UM,
                um_per_coordinate=UM_PER_COORDINATE,
                scale_bar_color="black",
                scale_bar_linewidth=0.8,
                scale_bar_each_panel=SCALE_BAR_EACH_PANEL,
                show_colorbar=SHOW_COLORBAR,
                colorbar_label=COLORBAR_LABEL,
                colorbar_width_mm=COLORBAR_WIDTH_MM,
                colorbar_height_mm=COLORBAR_HEIGHT_MM,
                title="",
                save_path=output_pdf,
                save_dpi=SAVE_DPI,
                show=SHOW_FIGURES,
            )

            manifest.append({
                "gene": gene,
                "region": str(region),
                "mode": mode,
                "clusters": str(cluster_selection),
                "layer": LAYER,
                "vmin": -limit,
                "vmax": limit,
                "n_samples": sum(r is not None for r in sample_results),
                "pdf": str(output_pdf),
            })

manifest_df = pd.DataFrame(manifest)
manifest_path = OUT_DIR / "D1_D2_combined_spatial_heatmap_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")
print(f"\nCompleted: {len(manifest_df)} PDF files")
print(f"Output directory: {OUT_DIR}")
print(f"Output manifest: {manifest_path}")


## Panels J-K | WT-reference anatomical gradients

Uses WT gene QC, fixed WT gene ordering, animal-level averaging, centered log2 expression, and three-bin display smoothing. Calls also export non-WT and per-sample outputs.


In [ ]:
# Imports and settings.
from pathlib import Path
import re,warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D


In [ ]:
H5AD_FILE=Path(r"F:\BASSFISH_HD\Source_data\integrated_5_str_with_axis_xy.h5ad")
OUT_ROOT=Path(r"F:\BASSFISH_HD\Figure\Figure_3\Panel\my_data\output_show_QC_v2")
CSV_DIR=OUT_ROOT/'sample_heatmap_csv'
FIG_DIR=OUT_ROOT/'sample_heatmap_pdf'
CONDITION_CSV_DIR=OUT_ROOT/'condition_mean_heatmap_csv'
CONDITION_FIG_DIR=OUT_ROOT/'condition_mean_heatmap_pdf'
PROFILE_CSV_DIR=OUT_ROOT/'three_category_profile_csv'
PROFILE_FIG_DIR=OUT_ROOT/'three_category_lineplot_pdf'
QC_DIR=OUT_ROOT/'qc'
for folder in [OUT_ROOT,CSV_DIR,FIG_DIR,CONDITION_CSV_DIR,CONDITION_FIG_DIR,PROFILE_CSV_DIR,PROFILE_FIG_DIR,QC_DIR]:
    folder.mkdir(parents=True,exist_ok=True)
CONDITION_MEAN_CONDITIONS=['WT','Heter','Homo']
LINE_PLOT_CONDITIONS=['WT','Heter','Homo']
CONDITIONS_TO_PLOT=None
SAMPLES_TO_PLOT=None
REFERENCE_MODE='WT_only'
CMAP='RdBu_r'; VMIN=-1.; VMAX=1.
BAD_VALUE_COLOR='#D9D9D9'
PANEL_WIDTH_MM=30; PANEL_HEIGHT_MM=50; COL_GAP_MM=4;ROW_GAP_MM=10
LEFT_MARGIN_MM=12;RIGHT_MARGIN_MM=12;TOP_MARGIN_MM=11;BOTTOM_MARGIN_MM=10
COLORBAR_WIDTH_MM=3;COLORBAR_GAP_MM=5
SHOW_CATEGORY_LINES=True;SHOW_FIGURES=False
DPI=600;FONT_NAME='Arial';FONT_PT=7
LINE_COMBINED_WIDTH_MM=125;LINE_COMBINED_HEIGHT_MM=48
LINE_SINGLE_WIDTH_MM=62;LINE_SINGLE_HEIGHT_MM=48
LINEWIDTH=0.9;LINE_ALPHA=0.95;LINE_SHOW_SEM=False;LINE_SEM_ALPHA=0.08
LINE_SHOW_ZERO_LINE=True;LINE_YMIN=None;LINE_YMAX=None;LINE_X_AS_PERCENT=False
LINE_SAVE_SEPARATE=True;LINE_SHOW_LEGEND=True;LINE_SHOW_FIGURES=False
LINE_CLASS_COLORS={'high_to_low':'#1f77b4','non_gradient':'#7f7f7f','low_to_high':'#d62728'}
LINE_AXIS_STYLES={'dorsal_x':'-','dorsal_y':'--','nac_x':'-.','nac_y':':','ot_x':(0,(5,1,1,1))}
LINE_REVERSE_AXIS_FOR_DISPLAY={a:False for a in ['dorsal_x','dorsal_y','nac_x','nac_y','ot_x']}
mpl.rcParams.update({'font.family':FONT_NAME,'font.size':FONT_PT,'axes.labelsize':FONT_PT,
 'axes.titlesize':FONT_PT,'xtick.labelsize':FONT_PT,'ytick.labelsize':FONT_PT,
 'legend.fontsize':FONT_PT,'pdf.fonttype':42,'ps.fonttype':42,'axes.linewidth':0.5})


In [ ]:
N_BINS = 10
BIN_CENTERS = np.arange(N_BINS, dtype=float) * (100.0 / N_BINS) + 50.0 / N_BINS
R_THRESHOLD = 0.5
MIN_VALID_BINS_FOR_R = 5
MIN_CELLS_PER_BIN = 5
QC_MIN_WT_GENE_MEAN = 1.0
QC_MIN_WT_DETECTION_FRAC = 0.02
QC_SAMPLE_DETECTION_FRAC = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.5
QC_MIN_VALID_REFERENCE_BINS = 8
QC_MIN_DIRECTIONAL_LOG2_RANGE = 0.0
PSEUDOCOUNT_FRAC_OF_WT_MEAN = 0.05
PSEUDOCOUNT_FLOOR_NORM1E4 = 0.1
SMOOTH_BINS = True

AXIS_CONFIG = {
 'dorsal_x':{'region':'d','orientation':'x','label':'Dorsal x'},
 'dorsal_y':{'region':'d','orientation':'y','label':'Dorsal y'},
 'nac_x':{'region':'n','orientation':'x','label':'NAc x'},
 'nac_y':{'region':'n','orientation':'y','label':'NAc y'},
 'ot_x':{'region':'o','orientation':'x','label':'OT x'},
}
AXIS_ORDER=list(AXIS_CONFIG)
CELLTYPE_REGEX={'MSN_D1':r'(?:^|_)D1(?:$|_)', 'MSN_D2':r'(?:^|_)D2(?:$|_)'}
CELLTYPE_ORDER=list(CELLTYPE_REGEX)
CATEGORY_ORDER=['high_to_low','non_gradient','low_to_high']
CONDITION_RENAME={'HD':'Heter'}

REGION_COL='region_str_manual'
CELLTYPE_COL='refined_cluster_4'
CONDITION_COL='condition'
SAMPLE_COL='sample'
AXIS_COLS={'x':'region_manual_axis_0_100','y':'region_manual_axis_y_0_100'}
LAYER='norm_1e4'

def mean_stack(mats):
    """Equally weight each biological animal; missing samples do not add zeros."""
    values=np.asarray(mats,dtype=float)
    finite=np.isfinite(values)
    counts=finite.sum(axis=0)
    sums=np.where(finite,values,0.0).sum(axis=0)
    mean=np.divide(sums,counts,out=np.full_like(sums,np.nan),where=counts>0)
    return mean, counts

def pearson_rows(mat):
    out=np.full(mat.shape[0],np.nan)
    for i,row in enumerate(mat):
        finite=np.isfinite(row)
        if finite.sum()<MIN_VALID_BINS_FOR_R: continue
        if np.nanstd(row[finite])==0:
            out[i]=0.
        else:
            out[i]=np.corrcoef(BIN_CENTERS[finite],row[finite])[0,1]
    return out

def get_cell_gene_metrics(indices):
    """Per-sample gene average and per-sample detected-cell fraction."""
    if len(indices)==0:
        n=adata.n_vars
        return np.full(n,np.nan), np.full(n,np.nan)
    Z=X[indices,:]
    if sparse.issparse(Z):
        means=np.asarray(Z.mean(axis=0)).ravel()
        detection=np.asarray(Z.getnnz(axis=0)).ravel()/len(indices)
    else:
        A=np.asarray(Z,dtype=float)
        means=np.nanmean(A,axis=0)
        detection=np.mean(A>0,axis=0)
    return np.asarray(means,float), np.asarray(detection,float)

def panel_masks(axis_key,celltype_name):
    a=axis_arrays[AXIS_CONFIG[axis_key]['orientation']]
    valid=(region_all==AXIS_CONFIG[axis_key]['region']) & np.isfinite(a) & (a>=0)&(a<=100)
    valid &= cluster_all.str.contains(CELLTYPE_REGEX[celltype_name],regex=True,na=False).to_numpy()
    bins=np.full(adata.n_obs,-1,dtype=int)
    bins[valid]=np.clip(np.floor(a[valid]/(100/N_BINS)),0,N_BINS-1).astype(int)
    return valid,bins

def animal_profile(mask,bins,cond,sample_name):
    selected=mask & (condition_all==cond) & (sample_all==sample_name)
    ix=np.flatnonzero(selected)
    global_mean,global_detect=get_cell_gene_metrics(ix)
    expr=np.full((adata.n_vars,N_BINS),np.nan,dtype=float)
    counts=np.zeros(N_BINS,dtype=int)
    for b in range(N_BINS):
        ib=np.flatnonzero(selected & (bins==b))
        counts[b]=len(ib)
        if len(ib)>=MIN_CELLS_PER_BIN:
            expr[:,b]=get_cell_gene_metrics(ib)[0]
    return {'raw':expr,'bin_counts':counts, 'global_mean':global_mean,
            'global_detection':global_detect,'n_cells':len(ix)}

def evaluate_wt_reference(records, genes, sample_keys=None):
    """Apply WT gene QC and classification.
    records maps sample names to animal profiles; all animals must be WT.
    """
    keys=list(records) if sample_keys is None else list(sample_keys)
    if not keys: raise ValueError('WT reference has zero biological samples')
    mats=[records[k]['raw'] for k in keys]
    wt_mean,contrib_counts=mean_stack(mats)
    wt_animal_means=np.stack([records[k]['global_mean'] for k in keys])
    wt_animal_dets=np.stack([records[k]['global_detection'] for k in keys])
    mean_expr,_=mean_stack(wt_animal_means)
    avg_det,_=mean_stack(wt_animal_dets)
    fraction_detected=np.mean(wt_animal_dets>=QC_SAMPLE_DETECTION_FRAC,axis=0)
    n_valid_bins=np.isfinite(wt_mean).sum(axis=1)
    r=pearson_rows(wt_mean)
    pseudo=np.maximum(PSEUDOCOUNT_FRAC_OF_WT_MEAN*np.nanmean(wt_mean,axis=1),
                      PSEUDOCOUNT_FLOOR_NORM1E4)

    wt_log=log2_transform(wt_mean,pseudo,center=False)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        amplitude=np.nanmax(wt_log,axis=1)-np.nanmin(wt_log,axis=1)
    qc_expr=np.isfinite(mean_expr)&(mean_expr>=QC_MIN_WT_GENE_MEAN)
    qc_detect=np.isfinite(avg_det)&(avg_det>=QC_MIN_WT_DETECTION_FRAC)
    qc_animals=fraction_detected>=QC_MIN_WT_SAMPLE_FRACTION
    qc_bins=n_valid_bins>=QC_MIN_VALID_REFERENCE_BINS
    qc_r=np.isfinite(r)
    ok=qc_expr & qc_detect & qc_animals & qc_bins & qc_r
    classes=np.full(len(genes),'excluded_QC',dtype=object)
    classes[ok]='non_gradient'
    directional=ok & (amplitude>=QC_MIN_DIRECTIONAL_LOG2_RANGE)
    classes[directional & (r<=-R_THRESHOLD)]='high_to_low'
    classes[directional & (r>=R_THRESHOLD)]='low_to_high'
    reasons=[]
    for i in range(len(genes)):
        parts=[]
        if not qc_expr[i]:parts.append('low_WT_expression')
        if not qc_detect[i]:parts.append('low_WT_detection')
        if not qc_animals[i]:parts.append('few_WT_animals_detecting')
        if not qc_bins[i]:parts.append('insufficient_spatial_bins')
        if not qc_r[i]:parts.append('invalid_pearson_r')
        reasons.append(';'.join(parts) if parts else '')
    table=pd.DataFrame({
        'gene':genes,'reference_axis_r':r, 'reference_class':classes,
        'qc_pass':ok,'qc_reason':reasons,
        'wt_mean_norm1e4':mean_expr,'wt_detection_fraction':avg_det,
        'wt_fraction_samples_detected':fraction_detected,
        'n_valid_wt_bins':n_valid_bins,'WT_log2_range':amplitude,
        'wt_pseudocount':pseudo,'n_reference_samples':len(keys),
    })
    return {'meta':table,'mean':wt_mean,'contributing_samples_per_bin':contrib_counts,
            'samples':keys}

def log2_transform(matrix, pseudocounts, center=True):
    """Per sample and gene: log2((bin+PC)/(10-bin mean+PC)), then row mean=0.
    NA bins remain NA. Fixed PC across genotypes is derived from full WT.
    """
    A=np.asarray(matrix,dtype=float)
    pc=np.asarray(pseudocounts,dtype=float).reshape(-1,1)
    valid=np.isfinite(A)
    n=valid.sum(axis=1,keepdims=True)
    rowmean=np.divide(np.where(valid,A,0.0).sum(axis=1,keepdims=True),n,
                       out=np.full((A.shape[0],1),np.nan),where=n>0)
    with np.errstate(divide='ignore',invalid='ignore'):
        L=np.log2((A+pc)/(rowmean+pc))
    L[~valid]=np.nan
    if center:
        non_na=np.isfinite(L)
        cnt=non_na.sum(axis=1,keepdims=True)
        mu=np.divide(np.where(non_na,L,0).sum(axis=1,keepdims=True),cnt,
                     out=np.full((len(L),1),np.nan),where=cnt>0)
        L=L-mu
    return L

def smooth_three_bins_edge_repeat(matrix):
    """3-bin mean of displayed log2 values; endpoints duplicated, NA center stays NA."""
    z=np.asarray(matrix,float)
    padded=np.pad(z,((0,0),(1,1)),mode='edge')
    windows=np.stack((padded[:,:-2],padded[:,1:-1],padded[:,2:]),axis=0)
    valid=np.isfinite(windows)
    counts=valid.sum(axis=0)
    sums=np.where(valid,windows,0).sum(axis=0)
    sm=np.divide(sums,counts,out=np.full(z.shape,np.nan),where=counts>0)
    sm[~np.isfinite(z)]=np.nan
    return sm

def sem_samples(values):
    a=np.asarray(values,float)
    valid=np.isfinite(a)
    n=valid.sum(axis=0)
    mu=np.divide(np.where(valid,a,0).sum(axis=0),n,
                 out=np.full(a.shape[1:],np.nan),where=n>0)
    delta=np.where(valid,a-mu,0.)
    variance=np.divide((delta**2).sum(axis=0),n-1,
                out=np.full(mu.shape,np.nan),where=n>1)
    sem=np.sqrt(variance/np.maximum(n,1))
    return mu,sem,n

def safe_filename(s):
    return re.sub(r'[<>:"/\\|?*\s]+','_',str(s)).strip('._') or 'unnamed'

def set_up_data():
    global adata,X,genes,region_all,cluster_all,condition_all,sample_all,axis_arrays
    if not H5AD_FILE.is_file():raise FileNotFoundError(H5AD_FILE)
    adata=sc.read_h5ad(H5AD_FILE)
    need=[REGION_COL,CELLTYPE_COL,CONDITION_COL,SAMPLE_COL,*AXIS_COLS.values()]
    absent=[c for c in need if c not in adata.obs]
    if absent:raise KeyError('Missing adata.obs columns: '+str(absent))
    if LAYER not in adata.layers:raise KeyError('Missing adata.layers: '+LAYER)
    X=adata.layers[LAYER]
    genes=np.asarray(adata.var_names.astype(str))
    if len(set(genes))!=len(genes):raise ValueError('Duplicate gene names in var_names')
    region_all=adata.obs[REGION_COL].astype(str).to_numpy()
    cluster_all=adata.obs[CELLTYPE_COL].astype(str)
    condition_all=adata.obs[CONDITION_COL].astype(str).replace(CONDITION_RENAME).to_numpy()
    sample_all=adata.obs[SAMPLE_COL].astype(str).to_numpy()
    axis_arrays={k:pd.to_numeric(adata.obs[v],errors='coerce').to_numpy(float)
                 for k,v in AXIS_COLS.items()}
    if not np.any(condition_all=='WT'):raise ValueError('No WT reference')
    print('Loaded',adata.n_obs,'cells,',adata.n_vars,'genes')
    print('Samples:',pd.DataFrame({'condition':condition_all,'sample':sample_all}).drop_duplicates().to_string(index=False))


In [ ]:
set_up_data()
all_sample_table=(pd.DataFrame({'condition':condition_all,'sample':sample_all})
                  .drop_duplicates().sort_values(['condition','sample']).reset_index(drop=True))
if not set(CONDITION_MEAN_CONDITIONS).issubset(set(all_sample_table.condition)):
    raise ValueError('Requested condition missing from H5AD; adjust CONDITION_MEAN_CONDITIONS')
plot_sample_table=all_sample_table.copy()
if CONDITIONS_TO_PLOT is not None:
    plot_sample_table=plot_sample_table[plot_sample_table.condition.isin(CONDITIONS_TO_PLOT)]
if SAMPLES_TO_PLOT is not None:
    plot_sample_table=plot_sample_table[plot_sample_table['sample'].isin(SAMPLES_TO_PLOT)]
if plot_sample_table.empty:raise ValueError('No samples selected to plot')
sample_table=plot_sample_table.copy()

panel_cache={}; sample_matrices={}; reference_by_panel={}
reference_rows=[];qc_bin_rows=[];sample_export_rows=[]
for axis_key in AXIS_ORDER:
    for ct in CELLTYPE_ORDER:
        mask,bins=panel_masks(axis_key,ct)
        panel_records={}
        for _,srow in all_sample_table.iterrows():
            cond,sname=str(srow.condition),str(srow['sample'])
            rec=animal_profile(mask,bins,cond,sname)
            panel_records[(cond,sname)]=rec
            for b,n in enumerate(rec['bin_counts']):
                qc_bin_rows.append({'axis':axis_key,'celltype':ct,'condition':cond,
                                    'sample':sname,'bin':b,'n_cells':int(n),
                                    'usable':bool(n>=MIN_CELLS_PER_BIN)})
        wt_samples=[str(s) for s in all_sample_table.loc[all_sample_table.condition.eq('WT'),'sample']
                    if panel_records[('WT',str(s))]['n_cells']>0]
        wt_profiles={s:panel_records[('WT',s)] for s in wt_samples}
        result=evaluate_wt_reference(wt_profiles,genes)
        met=result['meta'].copy()
        met.insert(0,'celltype',ct);met.insert(0,'axis',axis_key)
        reference_rows.append(met)
        accepted=met.loc[met.qc_pass].copy()
        inds=[];sections=[]
        for cat in CATEGORY_ORDER:
            group=accepted.loc[accepted.reference_class.eq(cat)].sort_values(
                ['reference_axis_r','gene'],ascending=[True,True])
            inds.extend(group.index.tolist());sections.append(len(inds))
        indices=np.asarray(inds,dtype=int)
        meta=met.iloc[indices].reset_index(drop=True)
        meta['row']=np.arange(1,len(meta)+1)
        if len(indices)==0:raise RuntimeError(f'{axis_key}/{ct}: all genes failed WT QC; review thresholds')
        ref={'order':indices,'meta':meta,'sections':sections,'WT_pseudocount':result['meta']['wt_pseudocount'].to_numpy()}
        reference_by_panel[(axis_key,ct)]=ref
        print(axis_key,ct,'WT QC pass',len(indices),'of',len(genes),'class counts',
              meta.reference_class.value_counts().to_dict())
        for key,rec in panel_records.items():
            cond,sname=key
            raw=rec['raw'][indices,:]
            heat_raw=log2_transform(raw,ref['WT_pseudocount'][indices],center=True)
            plotted=smooth_three_bins_edge_repeat(heat_raw) if SMOOTH_BINS else heat_raw.copy()
            sample_matrices[(cond,sname,axis_key,ct)]={'raw':raw,'heat_raw':heat_raw,
               'heat':plotted,'bin_counts':rec['bin_counts']}
            base=meta[['gene','reference_class','reference_axis_r']].rename(columns={'reference_class':'trend_category'}).copy()
            stem=f'{safe_filename(cond)}__{safe_filename(sname)}__{axis_key}__{ct}'
            saved={}
            for name,arr in [('raw_norm1e4',raw),('log2_centered_unsmoothed',heat_raw),
                             ('log2_centered_plotted',plotted)]:
                table=base.copy()
                for b in range(N_BINS):table[f'bin{b:02d}']=arr[:,b]
                path=CSV_DIR/f'{stem}__{name}.csv';table.to_csv(path,index=False)
                saved[name]=str(path)
            sample_export_rows.append({'condition':cond,'sample':sname,'axis':axis_key,
              'celltype':ct,**saved})

full_ref=pd.concat(reference_rows,ignore_index=True)
full_ref.to_csv(QC_DIR/'WT_full_reference_QC_all_genes.csv',index=False)
ordered_ref=pd.concat([x['meta'] for x in reference_by_panel.values()],ignore_index=True)
ordered_ref.to_csv(OUT_ROOT/'WT_fixed_gene_order_pass_QC.csv',index=False)
pd.DataFrame(qc_bin_rows).to_csv(QC_DIR/'per_sample_axis_bin_cell_counts.csv',index=False)
pd.DataFrame(sample_export_rows).to_csv(OUT_ROOT/'per_sample_heatmap_csv_manifest.csv',index=False)


In [ ]:
condition_matrices={}; condition_manifest_rows=[]
for cond in CONDITION_MEAN_CONDITIONS:
    animals=all_sample_table.loc[all_sample_table.condition.eq(cond),'sample'].astype(str).tolist()
    for axis_key in AXIS_ORDER:
        for ct in CELLTYPE_ORDER:
            arr=np.stack([sample_matrices[(cond,s,axis_key,ct)]['heat_raw'] for s in animals])
            mean_profile,sem,n=sem_samples(arr)
            smooth=smooth_three_bins_edge_repeat(mean_profile) if SMOOTH_BINS else mean_profile.copy()
            raw_mean,n_raw=mean_stack([sample_matrices[(cond,s,axis_key,ct)]['raw'] for s in animals])
            condition_matrices[(cond,axis_key,ct)]={'heat_raw':mean_profile,'heat':smooth,
              'sem':sem,'n_samples_per_gene_bin':n,'raw':raw_mean}
            meta=reference_by_panel[(axis_key,ct)]['meta']
            base=meta[['gene','reference_class','reference_axis_r']].rename(columns={'reference_class':'trend_category'}).copy()
            stem=f'{cond}__{axis_key}__{ct}'; paths={}
            for label,mat in [('raw_norm1e4_mean',raw_mean),('mean_log2_centered_unsmoothed',mean_profile),
              ('mean_log2_centered_plotted',smooth),('sem_log2_centered',sem),('n_animals_by_gene_bin',n)]:
                table=base.copy()
                for b in range(N_BINS):table[f'bin{b:02d}']=mat[:,b]
                path=CONDITION_CSV_DIR/f'{stem}__{label}.csv';table.to_csv(path,index=False)
                paths[label]=str(path)
            condition_manifest_rows.append({'condition':cond,'axis':axis_key,'celltype':ct,
             'n_animals':len(animals),**paths})
pd.DataFrame(condition_manifest_rows).to_csv(OUT_ROOT/'condition_mean_heatmap_matrix_manifest.csv',index=False)


In [ ]:
def plot_ten_panels(panel_values, plot_title, output_pdf):
    """panel_values[(axis_key, celltype)] = (n_genes, 10) matrix.

    Fixed WT order, fixed colormap and norm shared by ALL outputs.
    """
    ncols = len(AXIS_ORDER)
    nrows = len(CELLTYPE_ORDER)

    page_width_mm = (
        LEFT_MARGIN_MM
        + ncols * PANEL_WIDTH_MM
        + (ncols - 1) * COL_GAP_MM
        + COLORBAR_GAP_MM + COLORBAR_WIDTH_MM + RIGHT_MARGIN_MM
    )
    page_height_mm = (
        BOTTOM_MARGIN_MM + nrows * PANEL_HEIGHT_MM
        + (nrows - 1) * ROW_GAP_MM + TOP_MARGIN_MM
    )
    fig = plt.figure(
        figsize=(page_width_mm / 25.4, page_height_mm / 25.4),
        facecolor="white",
    )
    norm = TwoSlopeNorm(vmin=VMIN, vcenter=0, vmax=VMAX)
    cmap = mpl.colormaps[CMAP].copy()
    cmap.set_bad(BAD_VALUE_COLOR)

    image = None
    for row_idx, celltype_name in enumerate(CELLTYPE_ORDER):
        for col_idx, axis_key in enumerate(AXIS_ORDER):
            key = (axis_key, celltype_name)
            mat = panel_values[key]
            ref = reference_by_panel[key]
            sections = ref["sections"]
            n_genes = len(ref["meta"])
            if mat.shape != (n_genes, N_BINS):
                raise ValueError(f"Invalid matrix shape for {key}: {mat.shape}")
            left_mm = LEFT_MARGIN_MM + col_idx * (PANEL_WIDTH_MM + COL_GAP_MM)
            bottom_mm = BOTTOM_MARGIN_MM + (nrows - 1 - row_idx) * (
                PANEL_HEIGHT_MM + ROW_GAP_MM
            )
            ax = fig.add_axes([
                left_mm / page_width_mm,
                bottom_mm / page_height_mm,
                PANEL_WIDTH_MM / page_width_mm,
                PANEL_HEIGHT_MM / page_height_mm,
            ])
            image = ax.imshow(
                mat, aspect="auto", interpolation="nearest", cmap=cmap,
                norm=norm, origin="upper",
            )
            if SHOW_CATEGORY_LINES:
                for section in sections[:-1]:

                    ax.axhline(section - 0.5, color="black", linewidth=0.6)
            ax.set_xticks([0, 4.5, 9])
            ax.set_xticklabels(["0", "50", "100"])
            ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(AXIS_CONFIG[axis_key]["label"], pad=2)
            if col_idx == 0:
                ax.set_ylabel(celltype_name, labelpad=3)
            if row_idx == nrows - 1:
                ax.set_xlabel("Relative position")
            ax.tick_params(width=0.4, length=1.5, pad=1.5)
            for spine in ax.spines.values():
                spine.set_linewidth(0.4)

    cbar_x_mm = (
        LEFT_MARGIN_MM + ncols * PANEL_WIDTH_MM
        + (ncols - 1) * COL_GAP_MM + COLORBAR_GAP_MM
    )
    cbar_h_mm = PANEL_HEIGHT_MM
    cbar_bottom_mm = (page_height_mm - cbar_h_mm) / 2
    cax = fig.add_axes([
        cbar_x_mm / page_width_mm,
        cbar_bottom_mm / page_height_mm,
        COLORBAR_WIDTH_MM / page_width_mm,
        cbar_h_mm / page_height_mm,
    ])
    cb = fig.colorbar(image, cax=cax)
    cb.set_label("Centered log2 relative expression", labelpad=2)
    cb.ax.tick_params(width=0.4, length=1.5, pad=1.5)
    cb.outline.set_linewidth(0.4)

    fig.suptitle(plot_title, fontsize=FONT_PT, y=0.995)
    fig.savefig(
        output_pdf, dpi=DPI, facecolor="white", edgecolor="white",
        transparent=False, bbox_inches=None,
    )
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", output_pdf)
    return output_pdf


In [ ]:
smoothing_name = "smooth3_edge_repeat" if SMOOTH_BINS else "unsmoothed"
condition_pdf_rows = []

for condition in CONDITION_MEAN_CONDITIONS:
    panel_values = {
        (axis_key, celltype_name):
        condition_matrices[(condition, axis_key, celltype_name)]["heat"]
        for axis_key in AXIS_ORDER
        for celltype_name in CELLTYPE_ORDER
    }
    output_pdf = (
        CONDITION_FIG_DIR
        / f"{condition}_D1_D2_5axes_10heatmaps_{smoothing_name}.pdf"
    )
    pdf_path = plot_ten_panels(
        panel_values,
        f"{condition} | all biological samples | {smoothing_name}",
        output_pdf,
    )
    condition_pdf_rows.append({
        "condition": condition,
        "n_samples": int((all_sample_table["condition"] == condition).sum()),
        "reference_mode": REFERENCE_MODE,
        "vmin": VMIN,
        "vmax": VMAX,
        "smooth": SMOOTH_BINS,
        "pdf": str(pdf_path),
    })

pd.DataFrame(condition_pdf_rows).to_csv(
    OUT_ROOT / "condition_mean_heatmap_pdf_manifest.csv", index=False
)


In [ ]:
pdf_rows = []
for _, row in sample_table.iterrows():
    condition = str(row["condition"])
    sample_name = str(row["sample"])
    panel_values = {
        (axis_key, celltype_name):
        sample_matrices[(condition, sample_name, axis_key, celltype_name)]["heat"]
        for axis_key in AXIS_ORDER
        for celltype_name in CELLTYPE_ORDER
    }
    output_pdf = (
        FIG_DIR
        / (
            f"{safe_filename(condition)}__{safe_filename(sample_name)}__"
            f"D1_D2_5axes_10heatmaps_{smoothing_name}.pdf"
        )
    )
    pdf_path = plot_ten_panels(
        panel_values,
        f"{condition} | {sample_name} | {smoothing_name}",
        output_pdf,
    )
    pdf_rows.append({
        "condition": condition,
        "sample": sample_name,
        "reference_mode": REFERENCE_MODE,
        "three_bin_smoothing": SMOOTH_BINS,
        "color_vmin": VMIN,
        "color_vmax": VMAX,
        "pdf": str(pdf_path),
    })

pdf_manifest = pd.DataFrame(pdf_rows)
pdf_manifest.to_csv(OUT_ROOT / "per_sample_heatmap_manifest.csv", index=False)
print("Saved per-sample PDFs:", len(pdf_manifest))


In [ ]:
profile_sample_rows=[]; profile_summary_rows=[]
for axis_key in AXIS_ORDER:
    for ct in CELLTYPE_ORDER:
        ref=reference_by_panel[(axis_key,ct)]
        cats=ref['meta']['reference_class'].to_numpy()
        for cat in CATEGORY_ORDER:
            within=np.flatnonzero(cats==cat)
            for cond in CONDITION_MEAN_CONDITIONS:
                animals=all_sample_table.loc[all_sample_table.condition.eq(cond),'sample'].astype(str).tolist()
                profiles=[]
                for s in animals:
                    heat=sample_matrices[(cond,s,axis_key,ct)]['heat_raw']

                    sub=heat[within,:]
                    if len(within):
                        avg,genecount=mean_stack(sub)
                    else:
                        avg=np.full(N_BINS,np.nan)
                        genecount=np.zeros(N_BINS,dtype=int)
                    profiles.append(avg)
                    for b,val in enumerate(avg):
                        profile_sample_rows.append({'axis':axis_key,'celltype':ct,'trend_category':cat,
                            'condition':cond,'sample':s,'bin':b,'value':val,
                            'n_genes_total':len(within),'n_genes_valid':int(genecount[b])})
                mean,sem,n=sem_samples(np.stack(profiles))
                for b in range(N_BINS):
                    profile_summary_rows.append({'axis':axis_key,'celltype':ct,'trend_category':cat,
                        'condition':cond,'bin':b,'mean':mean[b],'sem':sem[b],
                        'n_samples':int(n[b]),'n_genes_total':len(within)})
profile_samples_df=pd.DataFrame(profile_sample_rows)
profile_summary_df=pd.DataFrame(profile_summary_rows)
profile_samples_df.to_csv(PROFILE_CSV_DIR/'all_10panels_sample_class_profiles_unsmoothed.csv',index=False)
profile_summary_df.to_csv(PROFILE_CSV_DIR/'all_10panels_class_profiles_unsmoothed_summary.csv',index=False)


In [ ]:
def plot_one_celltype_gene_classes(ax, data, celltype):
    sub_celltype = data.loc[data["celltype"].eq(celltype)]
    x = np.arange(N_BINS, dtype=float)
    if LINE_X_AS_PERCENT:
        x = np.asarray(BIN_CENTERS, dtype=float)
    for axis_key in AXIS_ORDER:
        sub_axis = sub_celltype.loc[sub_celltype["axis"].eq(axis_key)]
        for category in CATEGORY_ORDER:
            part = sub_axis.loc[sub_axis["trend_category"].eq(category)].sort_values("bin")
            if len(part) != N_BINS:
                raise ValueError(f"Unexpected 10-bin profile: {celltype}, {axis_key}, {category}")
            y = part["mean"].to_numpy(dtype=float)
            sem = part["sem"].to_numpy(dtype=float)
            if LINE_REVERSE_AXIS_FOR_DISPLAY.get(axis_key, False):
                y = y[::-1]
                sem = sem[::-1]
            ax.plot(
                x, y,
                color=LINE_CLASS_COLORS[category],
                linestyle=LINE_AXIS_STYLES[axis_key],
                linewidth=LINEWIDTH, alpha=LINE_ALPHA,
                zorder=3,
            )
            if LINE_SHOW_SEM:
                good = np.isfinite(y) & np.isfinite(sem)
                if good.any():
                    ax.fill_between(
                        x, y-sem, y+sem, where=good,
                        color=LINE_CLASS_COLORS[category],
                        alpha=LINE_SEM_ALPHA, linewidth=0, zorder=1,
                    )
    if LINE_SHOW_ZERO_LINE:
        ax.axhline(0, color="black", lw=0.5, alpha=0.35, zorder=0)
    if LINE_X_AS_PERCENT:
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 50, 100])
        ax.set_xlabel("Relative position")
    else:
        ax.set_xlim(-0.2, N_BINS-0.8)
        ax.set_xticks(range(N_BINS))
        ax.set_xlabel("Bin")
    if LINE_YMIN is not None or LINE_YMAX is not None:
        ax.set_ylim(bottom=LINE_YMIN, top=LINE_YMAX)
    ax.set_title(celltype, pad=2)
    ax.tick_params(axis="both", width=0.5, length=2, pad=1)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_linewidth(0.5)


def add_class_and_axis_legends(ax):
    categories = [
        Line2D([0], [0], color=LINE_CLASS_COLORS[c], lw=1.2, label=c)
        for c in CATEGORY_ORDER
    ]
    axes = [
        Line2D([0], [0], color="black", linestyle=LINE_AXIS_STYLES[a],
               lw=1.2, label=AXIS_CONFIG[a]["label"])
        for a in AXIS_ORDER
    ]
    leg1 = ax.legend(
        handles=categories, title="Gene category", frameon=False,
        loc="upper left", bbox_to_anchor=(1.03, 1.00),
        borderaxespad=0, labelspacing=0.35, handlelength=2.0,
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=axes, title="Spatial axis", frameon=False,
        loc="upper left", bbox_to_anchor=(1.03, 0.48),
        borderaxespad=0, labelspacing=0.35, handlelength=2.0,
    )


line_plot_manifest_rows = []
for cond in LINE_PLOT_CONDITIONS:
    if cond not in CONDITION_MEAN_CONDITIONS:
        raise ValueError(f"Line condition not present in condition-mean set: {cond}")
    cond_df = profile_summary_df.loc[profile_summary_df["condition"].eq(cond)].copy()
    if len(cond_df) != (len(AXIS_ORDER) * len(CELLTYPE_ORDER) *
                        len(CATEGORY_ORDER) * N_BINS):
        raise ValueError(f"Incomplete line-plot summary table for condition {cond}")
    cond_df.to_csv(
        PROFILE_CSV_DIR / f"{cond}_D1_D2_5axes_3categories_merged.csv", index=False
    )

    fig, axes = plt.subplots(
        1, 2, figsize=(LINE_COMBINED_WIDTH_MM/25.4, LINE_COMBINED_HEIGHT_MM/25.4),
        sharex=True, sharey=True,
    )
    for col_idx, ct in enumerate(CELLTYPE_ORDER):
        plot_one_celltype_gene_classes(axes[col_idx], cond_df, ct)
    axes[0].set_ylabel("Mean log2 enrichment")
    if LINE_SHOW_LEGEND:
        add_class_and_axis_legends(axes[-1])

    if LINE_SHOW_LEGEND:
        fig.subplots_adjust(left=0.11, right=0.69, bottom=0.19, top=0.90, wspace=0.30)
    else:
        fig.tight_layout(pad=0.5, w_pad=1.1)
    combined_path = PROFILE_FIG_DIR / f"{cond}_D1_D2_5axes_3categories_unsmoothed.pdf"
    fig.savefig(combined_path, dpi=DPI, facecolor="white", bbox_inches=None)
    if LINE_SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    line_plot_manifest_rows.append({
        "condition": cond, "celltype": "D1_D2", "pdf": str(combined_path),
        "smoothed": False,
    })

    if LINE_SAVE_SEPARATE:
        for ct in CELLTYPE_ORDER:
            fig_one, ax_one = plt.subplots(
                figsize=(LINE_SINGLE_WIDTH_MM/25.4, LINE_SINGLE_HEIGHT_MM/25.4),
            )
            plot_one_celltype_gene_classes(ax_one, cond_df, ct)
            ax_one.set_ylabel("Mean log2 enrichment")
            if LINE_SHOW_LEGEND:
                add_class_and_axis_legends(ax_one)
                fig_one.subplots_adjust(left=0.23, right=0.57, bottom=0.21, top=0.91)
            else:
                fig_one.tight_layout(pad=0.5)
            path_one = PROFILE_FIG_DIR / f"{cond}_{ct}_5axes_3categories_unsmoothed.pdf"
            fig_one.savefig(path_one, dpi=DPI, facecolor="white", bbox_inches=None)
            if LINE_SHOW_FIGURES:
                plt.show()
            plt.close(fig_one)
            line_plot_manifest_rows.append({
                "condition": cond, "celltype": ct, "pdf": str(path_one),
                "smoothed": False,
            })

pd.DataFrame(line_plot_manifest_rows).to_csv(
    OUT_ROOT / "three_category_lineplot_manifest.csv", index=False
)
print("Saved unsmoothed 3-class line-plot PDFs:", PROFILE_FIG_DIR)


In [ ]:
# Imports and settings.
params={'H5AD_FILE':str(H5AD_FILE),'OUT_ROOT':str(OUT_ROOT),'layer':LAYER,
 'R_THRESHOLD':R_THRESHOLD,'MIN_CELLS_PER_BIN':MIN_CELLS_PER_BIN,
 'QC_MIN_WT_GENE_MEAN':QC_MIN_WT_GENE_MEAN,'QC_MIN_WT_DETECTION_FRAC':QC_MIN_WT_DETECTION_FRAC,
 'QC_SAMPLE_DETECTION_FRAC':QC_SAMPLE_DETECTION_FRAC,
 'QC_MIN_WT_SAMPLE_FRACTION':QC_MIN_WT_SAMPLE_FRACTION,
 'QC_MIN_VALID_REFERENCE_BINS':QC_MIN_VALID_REFERENCE_BINS,
 'QC_MIN_DIRECTIONAL_LOG2_RANGE':QC_MIN_DIRECTIONAL_LOG2_RANGE,
 'PSEUDOCOUNT_FRAC_OF_WT_MEAN':PSEUDOCOUNT_FRAC_OF_WT_MEAN,
 'PSEUDOCOUNT_FLOOR_NORM1E4':PSEUDOCOUNT_FLOOR_NORM1E4,
 'SMOOTH_BINS':SMOOTH_BINS, 'method':'mean of per-animal centered log2 profiles'}
import json
(OUT_ROOT/'analysis_parameters.json').write_text(json.dumps(params,indent=2,ensure_ascii=False),encoding='utf-8')
print('DONE -- all outputs in',OUT_ROOT)
